# ENSTA · Séance 1
# Observer, représenter, apprendre

**Introduction à l’apprentissage profond · dernière année**  
**Version étudiante 5.0 · 17 septembre 2026 · Python / PyTorch**

Comment passer d’un ensemble de mesures à une règle capable de prédire quelque chose que nous n’avons pas encore observé ? Ce TP construit la réponse en plusieurs étapes. Nous regarderons d’abord les données, puis nous choisirons une famille de modèles. Nous examinerons ensuite le calcul qui permet de les ajuster, avant de décider comment évaluer leurs prédictions. Enfin, nous reprendrons cette démarche lorsque l’information disponible n’est plus une étiquette, mais une équation différentielle.

Le fil principal repose sur un même jeu de classification à deux coordonnées. Vous le retrouverez dans les **parties A, B, C et D** : les objets construits dans une partie serviront dans les suivantes. La **partie E** reprend le mécanisme d’apprentissage sur un oscillateur amorti. Le prélude d’Anscombe et les compléments du notebook initial sont également conservés.

| Parcours | Durée indicative | Ce que vous allez faire |
|:--|:--:|:--|
| [Prélude — quartet d’Anscombe](#anscombe) | 20 min | Indexer, résumer et représenter quatre jeux de données. |
| [A — Données et protocole](#tp-a) | 20 min | Calculer une transformation affine et préparer les données sans fuite. |
| [B — Représentation](#tp-b) | 30 min | Construire un MLP et le comparer à un modèle affine. |
| [C — Gradients et apprentissage](#tp-c) | 30 min | Vérifier des dérivées et écrire une époque d’entraînement. |
| [D — Sélection et test](#tp-d) | 25 min | Comparer les configurations, choisir sur validation, puis évaluer sur test. |
| [E — Information physique](#tp-e) | 30 min | Construire un résidu différentiel et contrôler la solution apprise. |
| [Ticket de sortie](#sortie) | 5 min | Relier les différentes étapes en six phrases. |
| [Complément — optimiseurs](#optimisateurs) | Hors séance | Examiner séparément mémoire, adaptation et décroissance des poids. |

Les parties A à E représentent **135 minutes de travail pratique**, alternées avec le cours. Le prélude représente **20 minutes supplémentaires**, à placer selon les indications de l’enseignant ; il n’est pas à ajouter implicitement au déroulé de cinq heures. Les durées sont des repères de conduite, non des délais à tenir pour chaque cellule.

**Une manière de travailler.** Avant chaque calcul, formulez une prévision courte : une forme de tenseur, une allure de courbe ou un effet attendu. Complétez ensuite le code demandé, exécutez le contrôle, puis comparez le résultat à votre prévision. En binôme, alternez les rôles : la personne qui n’écrit pas vérifie les dimensions et demande ce que le résultat permet réellement de conclure.


## Démarrage et lecture du notebook

Ouvrez votre copie dans un environnement Jupyter disposant d’un noyau Python. Exécutez d’abord les cellules de préparation ci-dessous. Le TP utilise le processeur ; aucun téléchargement de jeu de données n’est nécessaire. La bibliothèque Plotly est facultative et n’intervient que dans une seconde représentation du quartet d’Anscombe.

Vous rencontrerez trois types de cellules. Les cellules **fournies** préparent une expérience ou tracent une figure : exécutez-les sans les réécrire. Les cellules **à compléter** contiennent des emplacements `None` ou des commentaires numérotés : remplacez-les uniquement là où l’énoncé le demande. Les cellules de **contrôle** vérifient votre travail et affichent les grandeurs à interpréter.

Pour chaque exercice, lisez l’énoncé, complétez la cellule correspondante, puis exécutez son contrôle. Une définition de fonction ne lance pas encore son calcul : il faut aussi exécuter la cellule qui l’appelle. Après avoir corrigé une fonction, exécutez de nouveau sa définition avant de relancer le contrôle.

Les messages « exercice à compléter » indiquent simplement qu’un emplacement reste vide. Une autre erreur demande un diagnostic : commencez par les formes des tenseurs et l’ordre d’exécution. Ne modifiez pas les assertions pour les faire disparaître. Les aides repliables sont là pour débloquer une étape, pas pour remplacer votre prévision.

**À conserver.** Écrivez vos réponses dans les cellules « Votre trace de travail », en passant la cellule en mode édition. Pendant la séance, quelques mots ou une phrase argumentée suffisent. Sauvegardez régulièrement votre notebook ; son fichier et les variables présentes dans le noyau sont deux objets différents.


In [ ]:
# À décommenter seulement si nécessaire, puis redémarrer le noyau.
# %pip install torch numpy matplotlib scipy


In [ ]:
import copy
import math
import platform
import time
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader

SEED = 2026
DEVICE = torch.device("cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(1)  # petits tenseurs : limiter le surcoût du parallélisme
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
NOTEBOOK_START = time.perf_counter()
from IPython.display import display, Markdown

def export_figure(fig, name):
    # Option pour l'enseignant : définir ENSTA_FIGURE_DIR pour exporter les figures.
    destination = os.environ.get("ENSTA_FIGURE_DIR")
    if destination:
        directory = Path(destination)
        directory.mkdir(parents=True, exist_ok=True)
        fig.savefig(directory / f"{name}.pdf", bbox_inches="tight")
        fig.savefig(directory / f"{name}.png", dpi=160, bbox_inches="tight")
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        for index, axis in enumerate(fig.axes, start=1):
            bbox = axis.get_tightbbox(renderer).transformed(fig.dpi_scale_trans.inverted()).expanded(1.05, 1.10)
            # Masquer les axes voisins pour que leurs textes ne débordent pas dans le recadrage.
            visibility = [other.get_visible() for other in fig.axes]
            for other in fig.axes:
                other.set_visible(other is axis)
            fig.savefig(directory / f"{name}_panel{index}.pdf", bbox_inches=bbox)
            fig.savefig(directory / f"{name}_panel{index}.png", dpi=160, bbox_inches=bbox)
            for other, was_visible in zip(fig.axes, visibility):
                other.set_visible(was_visible)
print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | {DEVICE}")
print("Les graines fixées facilitent la comparaison ; une identité bit à bit entre machines n'est pas garantie.")


def verifier_completion(exercice, **valeurs):
    """Signaler un emplacement non rempli avant qu'il ne provoque une erreur de calcul."""
    manquants = [nom for nom, valeur in valeurs.items() if valeur is None]
    if manquants:
        raise NotImplementedError(
            f"{exercice} — Il reste à compléter : {', '.join(manquants)}. "
            "Reprenez les étapes de l'énoncé, remplacez les valeurs None, "
            "puis exécutez de nouveau la définition et son contrôle."
        )

print(f"NumPy {np.__version__}")


<a id="anscombe"></a>
## Prélude — Regarder les données avant de choisir un modèle
**Quartet d’Anscombe · 20 minutes**

Imaginez que quatre personnes vous confient chacune onze mesures et vous demandent si une droite décrit convenablement leurs données. Vous pourriez commencer par calculer une moyenne, une variance et une corrélation. Mais ces nombres suffiraient-ils à donner le même conseil aux quatre personnes ? Le quartet d’Anscombe, introduit dans l’article *Graphs in Statistical Analysis* cité en fin de notebook, permet de poser cette question sur des observations que l’on peut toutes examiner.

Nous allons d’abord lire le tableau, puis calculer les mêmes résumés pour chaque jeu, et seulement ensuite regarder les figures. **Ne lancez pas encore les cellules de représentation.** L’intérêt de l’expérience est de conserver la différence entre votre conclusion après le tableau et votre conclusion après les graphiques.

La cellule fournie rassemble les quatre jeux dans `ANSCOMBE`. Sa forme est `(4, 11, 2)` : le premier axe désigne le jeu, le deuxième l’observation, le troisième la variable. Sur ce dernier axe, l’indice `0` correspond à $x$ et l’indice `1` à $y$. Une ligne de deux nombres représente donc un couple mesuré, non deux exemples indépendants.


In [ ]:
ANSCOMBE = torch.tensor([
    [[10., 8.04], [8., 6.95], [13., 7.58], [9., 8.81], [11., 8.33],
     [14., 9.96], [6., 7.24], [4., 4.26], [12., 10.84], [7., 4.82], [5., 5.68]],
    [[10., 9.14], [8., 8.14], [13., 8.74], [9., 8.77], [11., 9.26],
     [14., 8.10], [6., 6.13], [4., 3.10], [12., 9.13], [7., 7.26], [5., 4.74]],
    [[10., 7.46], [8., 6.77], [13., 12.74], [9., 7.11], [11., 7.81],
     [14., 8.84], [6., 6.08], [4., 5.39], [12., 8.15], [7., 6.42], [5., 5.73]],
    [[8., 6.58], [8., 5.76], [8., 7.71], [8., 8.84], [8., 8.47],
     [8., 7.04], [8., 5.25], [19., 12.50], [8., 5.56], [8., 7.91], [8., 6.89]],
], dtype=torch.float64)
ANSCOMBE_LABELS = ("I", "II", "III", "IV")

print("Forme du tenseur complet :", tuple(ANSCOMBE.shape))
print("Un jeu de données :", tuple(ANSCOMBE[0].shape))
print("Une observation :", tuple(ANSCOMBE[0, 0].shape), "->", ANSCOMBE[0, 0].tolist())


### 0.1 — Retrouver une information dans le tenseur · 4 min

Commencez par relire le premier couple affiché. L’expression `ANSCOMBE[0, 0]` fixe le jeu et l’observation ; elle conserve les deux variables. Nous allons appliquer la même lecture à trois extractions.

**D’abord, sur papier ou oralement**, indiquez pour chacune les axes que vous conservez et ceux que vous fixez : le premier jeu complet ; les abscisses de tous les jeux ; les ordonnées du troisième jeu. Déduisez-en le nombre d’axes du résultat.

**Puis, dans la cellule à compléter**, affectez ces trois extractions à `jeu_I`, `tous_les_x` et `y_jeu_III`. Utilisez l’indexation, sans boucle. Exécutez le contrôle et comparez les formes obtenues à votre prévision. Les formes visées sont respectivement `(11, 2)`, `(4, 11)` et `(11,)`.

**À vérifier dans votre explication :** pourquoi le troisième jeu est-il désigné par l’indice `2` ? Que signifie le symbole `:` dans une position d’indexation ?


<details>
<summary>Un indice sur les axes</summary>

Une valeur entière sélectionne une position et retire cet axe du résultat. Le symbole `:` conserve toutes les positions d’un axe. L’indexation commence à zéro : le troisième élément a donc l’indice `2`. Repérez d’abord le dernier axe pour distinguer abscisses et ordonnées.

</details>


In [ ]:
# À compléter — une expression d'indexation pour chaque ligne.
jeu_I = None          # Premier jeu : conserver les observations et les deux variables.
tous_les_x = None      # Tous les jeux, toutes les observations, uniquement x.
y_jeu_III = None       # Troisième jeu, toutes les observations, uniquement y.


In [ ]:
verifier_completion("0.1", jeu_I=jeu_I, tous_les_x=tous_les_x, y_jeu_III=y_jeu_III)
assert jeu_I.shape == (11, 2), "jeu_I doit conserver les observations et les variables."
assert tous_les_x.shape == (4, 11), "tous_les_x doit conserver les jeux et les observations."
assert y_jeu_III.shape == (11,), "y_jeu_III doit conserver uniquement l'axe des observations."
assert torch.equal(jeu_I, ANSCOMBE[0]), "Vérifiez le jeu sélectionné."
assert torch.equal(tous_les_x, ANSCOMBE[..., 0]), "Vérifiez la variable sélectionnée."
assert torch.equal(y_jeu_III, ANSCOMBE[2, :, 1]), "Vérifiez le jeu et la variable sélectionnés."
print("Contrôle 0.1 réussi :", tuple(jeu_I.shape), tuple(tous_les_x.shape), tuple(y_jeu_III.shape))


### 0.2 — Construire le tableau des résumés · 7 min

Nous voulons maintenant appliquer le même raisonnement aux quatre jeux, sans copier quatre fois le programme. La fonction `summarize_anscombe` reçoit un tenseur `(J, N, 2)` et doit renvoyer une matrice `(J, 7)`. Chaque ligne contiendra, dans cet ordre, la moyenne de $x$, la moyenne de $y$, leurs deux variances, la corrélation, la pente de la droite et son ordonnée à l’origine.

**Étape 1 — Centrer les observations.** L’extraction de `x` et de `y` est fournie ; chacune a la forme `(J, N)`. Complétez `mean_x` et `mean_y` en calculant une moyenne par jeu. Chaque résultat doit avoir la forme `(J,)`. Le centrage qui suit est fourni : l’ajout de `[:, None]` remet la moyenne sous forme de colonne pour la soustraire à toutes les observations du même jeu.

**Étape 2 — Mesurer la dispersion et les variations communes.** À partir de `x_centered` et `y_centered`, complétez les deux variances et la covariance. Nous adoptons ici les statistiques d’échantillon, avec le dénominateur $N-1$ :

$$s_x^2=\frac{1}{N-1}\sum_i(x_i-\bar x)^2,\qquad
s_y^2=\frac{1}{N-1}\sum_i(y_i-\bar y)^2,\qquad
s_{xy}=\frac{1}{N-1}\sum_i(x_i-\bar x)(y_i-\bar y).$$

**Étape 3 — Passer à la corrélation et à la droite ajustée.** Utilisez les quantités précédentes pour compléter

$$r=\frac{s_{xy}}{\sqrt{s_x^2s_y^2}},\qquad a=\frac{s_{xy}}{s_x^2},\qquad b=\bar y-a\bar x.$$

L’assemblage des sept colonnes est fourni. Exécutez le tableau et son contrôle. Avant de poursuivre, formulez une phrase répondant à cette question : **sur la seule base de ces nombres, conseilleriez-vous des modèles différents pour les quatre jeux ?**


<details>
<summary>Aide de syntaxe : agréger le bon axe</summary>

Dans une matrice `(J, N)`, l’axe `1` parcourt les observations d’un même jeu. Les méthodes `.mean(dim=1)` et `.sum(dim=1)` rendent donc un nombre par jeu. `.square()` calcule les carrés terme à terme ; `x_centered * y_centered` calcule les produits des écarts correspondants. Ici, il faut bien diviser les sommes par `N-1`, et non prendre directement leur moyenne.

</details>


In [ ]:
def summarize_anscombe(data):
    """Renvoyer les sept résumés de chaque jeu, sous la forme (J, 7)."""
    x, y = data[:, :, 0], data[:, :, 1]  # Deux matrices (J, N).
    n = x.shape[1]

    # Étape 1 : une moyenne par jeu ; chaque résultat a la forme (J,).
    mean_x = None
    mean_y = None
    verifier_completion("0.2 — étape 1", mean_x=mean_x, mean_y=mean_y)
    x_centered = x - mean_x[:, None]
    y_centered = y - mean_y[:, None]

    # Étape 2 : sommes sur les observations, puis division par n - 1.
    var_x = None
    var_y = None
    covariance = None
    verifier_completion("0.2 — étape 2", var_x=var_x, var_y=var_y, covariance=covariance)

    # Étape 3 : utiliser les moments précédents, sans réajuster les données.
    correlation = None
    slope = None
    intercept = None
    verifier_completion("0.2 — étape 3", correlation=correlation, slope=slope, intercept=intercept)
    return torch.stack([mean_x, mean_y, var_x, var_y, correlation, slope, intercept], dim=1)


In [ ]:
anscombe_stats = summarize_anscombe(ANSCOMBE)
assert anscombe_stats.shape == (4, 7), "Une ligne par jeu, sept colonnes dans l'ordre demandé."
assert torch.isfinite(anscombe_stats).all(), "Une statistique n'est pas finie : vérifiez les dénominateurs."
columns = ("moy. x", "moy. y", "var. x", "var. y", "corr.", "pente", "ord. orig.")
print("jeu | " + " | ".join(f"{name:>9s}" for name in columns))
print("-" * 88)
for label, row in zip(ANSCOMBE_LABELS, anscombe_stats):
    print(f" {label:>2s} | " + " | ".join(f"{value.item():9.4f}" for value in row))

# Les nombres ne sont pas exactement identiques à cause de l'arrondi des données originales,
# mais ils sont suffisamment proches pour conduire au même résumé verbal.
assert torch.max(torch.abs(anscombe_stats[:, 0] - 9.0)) < 1e-12
assert torch.max(torch.abs(anscombe_stats[:, 1] - 7.5)) < 1e-3
assert torch.max(torch.abs(anscombe_stats[:, 5] - 0.5)) < 5e-4


#### Votre trace de travail

**Ce que le tableau suggère, avant de regarder les figures** — À compléter.


### 0.3 — Confronter le résumé aux observations · 6 min

Vous pouvez maintenant exécuter la représentation fournie. Les quatre panneaux ont les mêmes limites d’axes et montrent chacun la droite que vous venez de calculer. Les petits nombres sont les indices des observations ; ils permettront de désigner précisément un point dans votre réponse.

Commencez par comparer les jeux I et II : les points se répartissent-ils de la même manière autour de la droite ? Regardez ensuite le jeu III : quelle observation s’écarte le plus de la tendance suivie par les autres ? Terminez par le jeu IV : combien d’abscisses différentes les mesures contiennent-elles ?

La seconde cellule permet, facultativement, d’ouvrir la version Plotly en mettant `AFFICHER_VERSION_INTERACTIVE` à `True`. Le survol facilite la lecture des coordonnées. La figure Matplotlib suffit pour répondre à toutes les questions.


In [ ]:
def plot_anscombe(data, stats):
    fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharex=True, sharey=True,
                             constrained_layout=True)
    x_line = torch.linspace(3.0, 20.0, 200, dtype=data.dtype)

    for j, (label, ax) in enumerate(zip(ANSCOMBE_LABELS, axes.flat)):
        x = data[j, :, 0]
        y = data[j, :, 1]
        slope, intercept = stats[j, 5], stats[j, 6]
        ax.scatter(x.numpy(), y.numpy(), s=42, edgecolor="white", linewidth=0.8)
        ax.plot(x_line.numpy(), (slope * x_line + intercept).numpy(), linewidth=1.6)
        for i, (xi, yi) in enumerate(zip(x, y)):
            ax.annotate(str(i), (xi.item(), yi.item()), xytext=(4, 3),
                        textcoords="offset points", fontsize=7, alpha=0.7)
        ax.set_title(f"Jeu {label} — r = {stats[j, 4].item():.3f}")
        ax.set_xlim(3, 20)
        ax.set_ylim(2, 14)
        ax.grid(alpha=0.25)
        ax.set_xlabel("x")
        ax.set_ylabel("y")

    fig.suptitle("Quartet d’Anscombe — mêmes résumés, structures différentes", fontsize=13)
    export_figure(fig, "anscombe_quartet")
    plt.show()
    return fig

anscombe_figure = plot_anscombe(ANSCOMBE, anscombe_stats)


In [ ]:
AFFICHER_VERSION_INTERACTIVE = False

if AFFICHER_VERSION_INTERACTIVE:
    # Visualisation interactive facultative : survolez les points, zoomez et comparez les panneaux.
    try:
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
    except ImportError:
        print("Plotly n'est pas installé : la figure Matplotlib suffit pour poursuivre.")
    else:
        fig_interactive = make_subplots(
            rows=2, cols=2,
            subplot_titles=[f"Jeu {label}" for label in ANSCOMBE_LABELS],
            shared_xaxes=True, shared_yaxes=True,
            horizontal_spacing=0.09, vertical_spacing=0.12,
        )
        x_line_np = np.linspace(3.0, 20.0, 200)
        for j, label in enumerate(ANSCOMBE_LABELS):
            row, col = divmod(j, 2)
            row, col = row + 1, col + 1
            x_np = ANSCOMBE[j, :, 0].numpy()
            y_np = ANSCOMBE[j, :, 1].numpy()
            slope = anscombe_stats[j, 5].item()
            intercept = anscombe_stats[j, 6].item()
            fig_interactive.add_trace(
                go.Scatter(
                    x=x_np, y=y_np, mode="markers+text",
                    text=[str(i) for i in range(len(x_np))], textposition="top center",
                    customdata=np.arange(len(x_np)),
                    hovertemplate="point %{customdata}<br>x=%{x:.2f}<br>y=%{y:.2f}<extra></extra>",
                    name=f"Jeu {label}", showlegend=False,
                ), row=row, col=col,
            )
            fig_interactive.add_trace(
                go.Scatter(
                    x=x_line_np, y=slope * x_line_np + intercept,
                    mode="lines", hoverinfo="skip", showlegend=False,
                ), row=row, col=col,
            )
        fig_interactive.update_xaxes(range=[3, 20], title_text="x")
        fig_interactive.update_yaxes(range=[2, 14], title_text="y")
        fig_interactive.update_layout(
            height=650, width=900,
            title="Quartet d’Anscombe — exploration interactive",
            template="plotly_white",
        )
        fig_interactive.show()

else:
    print("Exploration interactive désactivée ; la figure précédente suffit pour poursuivre.")


### 0.4 — Formuler un diagnostic, sans décider trop vite · 3 min

Reprenez votre première conclusion à la lumière des graphiques. Pour chaque jeu, donnez **une observation précise**, puis **une conséquence pour l’usage d’une droite**. Vous pouvez répondre oralement aux quatre premières questions et ne rédiger que la conclusion finale.

1. Dans le jeu I, une tendance affine semble-t-elle plausible ? Quelle dispersion reste visible ?
2. Dans le jeu II, quelle structure la droite et la corrélation ne décrivent-elles pas ?
3. Dans le jeu III, quel point mérite un examen particulier ? Quel contrôle proposeriez-vous avant de décider de le retirer ?
4. Dans le jeu IV, que deviendraient la variance de $x$ et le calcul de la pente sans le point d’abscisse 19 ?
5. Quelle précaution conserveriez-vous avec des données de grande dimension que l’on ne peut plus représenter intégralement ?

**Pour passer à A :** vous devez pouvoir expliquer pourquoi un graphique complète un résumé numérique, sans en conclure que les statistiques sont inutiles.


#### Votre trace de travail

**Observations sur les jeux I et II** — À compléter.

**Observations sur les jeux III et IV** — À compléter.

**Précaution à retenir** — À compléter.


<a id="tp-a"></a>
## A — Des observations à un protocole expérimental
**20 minutes**

Le prélude nous a appris à ne pas confondre un résumé et les observations qu’il décrit. Nous passons maintenant à un échantillon plus grand, destiné à l’apprentissage. Avant de construire un réseau, nous devons donner un rôle à chaque observation et préciser les opérations que nous allons lui appliquer.

### A0 — Lire les données et leur répartition · 5 min

Chaque exemple est un point $x=(x_1,x_2)$, tiré uniformément dans le carré $[-1{,}6;1{,}6]^2$. Sans bruit, sa classe vaut `1` lorsque $x_1x_2>0$ et `0` sinon. Les quadrants alternent donc entre les deux classes : c’est la difficulté géométrique de XOR, avec une autre convention de noms de classes. Une étiquette est ensuite inversée aléatoirement avec une probabilité de 8 %.

**Avant d’exécuter la cellule**, dessinez les axes et indiquez la classe attendue dans chaque quadrant. Imaginez l’effet de quelques inversions d’étiquettes. **Puis exécutez la cellule fournie** : elle génère 900 observations et en réserve 540 à l’entraînement, 180 à la validation et 180 au test. Seul l’entraînement est représenté.

Repérez dans le code les trois ensembles d’indices, puis vérifiez les effectifs affichés. Sur le graphique, recherchez des points dont l’étiquette diffère de celle du quadrant. Vous n’avez pas à les corriger : ils font partie du problème que nous avons défini.

**Question de lecture.** Pourquoi la connaissance de la règle géométrique ne permet-elle pas de prédire toutes les étiquettes ? Une probabilité d’inversion de 8 % impose-t-elle exactement 8 % d’inversions dans cet échantillon ?

Les données de test sont conservées dans `TEST_SCELLE`. Ne les utilisez ni pour une figure ni pour un réglage avant D2. Ce nom rappelle une règle de travail ; il ne constitue pas une protection informatique.


In [ ]:
def make_xor(n=900, seed=SEED, flip_probability=0.08):
    g = torch.Generator().manual_seed(seed)
    x = 3.2 * torch.rand(n, 2, generator=g) - 1.6
    y_clean = (x[:, 0] * x[:, 1] > 0).long()
    flips = torch.rand(n, generator=g) < flip_probability
    y = torch.logical_xor(y_clean.bool(), flips).long()
    return x, y

X_all, y_all = make_xor()
g_split = torch.Generator().manual_seed(SEED + 1)
indices = torch.randperm(len(X_all), generator=g_split)
id_train, id_val, id_test = indices[:540], indices[540:720], indices[720:]
X_train_raw, y_train = X_all[id_train], y_all[id_train]
X_val_raw, y_val = X_all[id_val], y_all[id_val]
# Le test est scellé : aucun score, graphique ou réglage avant la fin de D.
TEST_SCELLE = (X_all[id_test].clone(), y_all[id_test].clone())
del X_all, y_all
assert set(id_train.tolist()).isdisjoint(id_val.tolist())
assert set(id_train.tolist()).isdisjoint(id_test.tolist())
assert set(id_val.tolist()).isdisjoint(id_test.tolist())
print("Apprentissage / validation / test :", len(id_train), len(id_val), len(id_test))
print("Équilibre des classes (entraînement seulement) :", torch.bincount(y_train).tolist())

fig, ax = plt.subplots(figsize=(5.2, 4.1))
ax.scatter(X_train_raw[:, 0], X_train_raw[:, 1], c=y_train, cmap="coolwarm", s=12, alpha=.7)
ax.set(xlabel="$x_1$", ylabel="$x_2$", title="Échantillon d'apprentissage : XOR bruité", aspect="equal")
export_figure(fig, "classification_data")
plt.show()


### A1 — Calculer les scores d’un lot d’observations · 5 min

Un premier modèle attribue des scores aux observations par une transformation affine. Nous allons écrire cette transformation directement, pour relier le calcul matriciel aux dimensions des données.

Dans **cet exercice et dans C1**, nous stockons les observations en lignes et les poids en colonnes :

$$X\in\mathbb{R}^{N\times d},\qquad W\in\mathbb{R}^{d\times C},\qquad
b\in\mathbb{R}^{C},\qquad Z=XW+b.$$

**1. Préparer le calcul.** Déduisez la forme de $Z$. Pour un exemple $i$ et un score $j$, écrivez la somme qui donne $Z_{ij}$. Cette écriture doit faire apparaître quelles composantes de l’exemple rencontrent quels poids.

**2. Compléter la fonction.** Remplacez `scores = None` par une expression qui calcule tous les scores à la fois. Utilisez le produit matriciel, puis l’addition du biais, sans boucle sur les observations.

**3. Vérifier et relier à PyTorch.** Exécutez la cellule de contrôle. Recalculez à la main le premier score de la première observation, puis comparez la forme de `w` à celle des poids stockés par `nn.Linear(2, 3)`. Attention : PyTorch stocke cette dernière matrice sous la forme `(C, d)`, transposée par rapport à la convention choisie ici.


<details>
<summary>Aide : produit matriciel et biais</summary>

L’opérateur `@` effectue un produit matriciel ; `*` effectue un produit terme à terme. Après le produit, vous devez avoir une matrice `(N, C)`. Ajouter un vecteur `(C,)` ajoute le même biais à chaque ligne : c’est la diffusion des dimensions, ou *broadcasting*.

</details>


In [ ]:
def affine(x, w, b):
    """Calculer les scores (N, C) à partir de x (N, d), w (d, C) et b (C,)."""
    # À compléter : produit matriciel, puis addition du biais.
    scores = None
    verifier_completion("A1", scores=scores)
    return scores


In [ ]:
a = torch.tensor([[1., 2.], [3., 4.], [-1., 2.]])
w = torch.tensor([[1., 0., -1.], [2., 1., 0.]])
b = torch.tensor([0.5, -0.5, 1.])
z = affine(a, w, b)
assert z.shape == (3, 3)
assert torch.allclose(z[0], torch.tensor([5.5, 1.5, 0.]))
print("Forme des scores :", tuple(z.shape))
print("Stockage nn.Linear(2, 3).weight :", tuple(nn.Linear(2, 3).weight.shape))

assert torch.allclose(z, a @ w + b), "Vérifiez toutes les lignes, pas seulement la première."
print("Les scores calculés :\n", z)


### A2 — Estimer une transformation, puis la réutiliser · 10 min

Les deux coordonnées vont être centrées et réduites. Cette opération possède elle-même des paramètres : une moyenne et une échelle pour chaque colonne. Nous les estimerons sur les observations d’entraînement, puis nous appliquerons **la même transformation** aux autres ensembles.

**1. Identifier l’axe.** `X_train_raw` a la forme `(540, 2)`. Nous voulons deux moyennes, et non une moyenne par observation. Quel axe faut-il agréger ? Pourquoi garder cet axe avec `keepdim=True` conduit-il à une forme `(1, 2)` utile pour la suite ?

**2. Compléter `fit_standardizer`.** Calculez `mean` et `std` colonne par colonne, en conservant cette forme `(1, d)`. Utilisez `unbiased=False` pour l’écart type : ici, nous décrivons la dispersion des données d’entraînement avec le dénominateur $N$, contrairement aux variances d’échantillon du prélude. Le bornage fourni après votre calcul évite une division par zéro lorsqu’une colonne est constante.

**3. Appliquer sans réestimer.** Les deux lignes qui transforment l’entraînement et la validation sont fournies. Lisez-les avant de les exécuter : elles doivent employer les mêmes `mean_train` et `std_train`. Aucune nouvelle estimation ne doit être faite sur la validation.

**4. Interpréter le contrôle.** Sur les deux colonnes de cet entraînement, la moyenne transformée doit être proche de zéro et l’écart type proche de un. Regardez ensuite la moyenne de validation : son éventuel écart à zéro est-il, à lui seul, une erreur de code ? Expliquez pourquoi il ne faut pas le « corriger » en recentrant séparément la validation.


<details>
<summary>Aide : deux statistiques par colonne</summary>

L’axe `0` parcourt les observations de `x_train`. Vous pouvez utiliser `.mean(dim=0, keepdim=True)` et `.std(dim=0, unbiased=False, keepdim=True)`. La fonction renvoie les deux statistiques ; elle ne doit pas modifier son entrée et ne doit pas consulter les variables de validation ou de test.

</details>


In [ ]:
def fit_standardizer(x_train):
    """Estimer sur l'entraînement une moyenne et une échelle de forme (1, d)."""
    # À compléter : deux statistiques colonne par colonne, sans changer de données.
    mean = None
    std = None
    verifier_completion("A2", mean=mean, std=std)
    std = std.clamp_min(1e-6)  # Fourni : éviter une division par zéro.
    return mean, std

# Fourni : apprendre la transformation une seule fois, puis la réutiliser.
mean_train, std_train = fit_standardizer(X_train_raw)
X_train = (X_train_raw - mean_train) / std_train
X_val = (X_val_raw - mean_train) / std_train


In [ ]:
assert mean_train.shape == std_train.shape == (1, 2)
assert torch.allclose(X_train.mean(0), torch.zeros(2), atol=1e-6)
assert torch.allclose(X_train.std(0, unbiased=False), torch.ones(2), atol=1e-6)
assert y_train.dtype == torch.long
print("Moyenne d’entraînement après transformation :", X_train.mean(0).tolist())
print("Moyenne validation après transformation :", X_val.mean(0).tolist())
print("Entrées :", X_train.dtype, tuple(X_train.shape), "| cibles :", y_train.dtype, tuple(y_train.shape))

# Contrôle supplémentaire fourni : une colonne constante doit rester calculable.
constant_probe = torch.tensor([[3., 1.], [3., 5.]])
constant_mean, constant_std = fit_standardizer(constant_probe)
constant_scaled = (constant_probe - constant_mean) / constant_std
assert torch.isfinite(constant_scaled).all(), "Une colonne constante ne doit pas créer de NaN."
assert torch.allclose(constant_scaled[:, 0], torch.zeros(2))
print("Colonne constante : calcul fini, valeurs centrées nulles.")


### Bilan A — Ce que nous pouvons maintenant réutiliser

Vous disposez de `X_train`, `y_train`, `X_val` et `y_val`, ainsi que de la transformation apprise. Avant de passer au réseau, répondez aux trois questions suivantes ; une courte conclusion écrite suffit après la discussion.

1. Quelles formes relient les observations, les poids, le biais et les scores dans A1 ? Pourquoi le biais est-il partagé entre les lignes ?
2. En quel sens les statistiques de standardisation sont-elles apprises ? À quel endroit une fuite apparaîtrait-elle si l’on utilisait toutes les données ?
3. Quelle propriété de la simulation autorise ici une partition aléatoire ? Donnez un type de données pour lequel il faudrait organiser autrement le partage.


#### Votre trace de travail

**Conclusion sur les dimensions** — À compléter.

**Conclusion sur le prétraitement et le partage** — À compléter.


<a id="tp-b"></a>
## B — De la frontière affine à la représentation apprise
**30 minutes**

Le protocole est fixé. Nous pouvons maintenant demander ce qu’un modèle est capable de représenter. Une droite pourra-t-elle séparer les classes présentes dans des quadrants alternés ? Et que change l’introduction de couches non linéaires ? Nous comparerons deux familles sur les mêmes observations, avant d’étudier en C le calcul qui ajuste leurs paramètres.

### B1 — Construire le réseau et vérifier son interface · 8 min

Le modèle doit produire **deux scores réels par observation**, un pour chaque classe. Ces scores sont les *logits* ; ils ne sont pas encore des probabilités. La perte `CrossEntropyLoss` reçoit directement les logits et les indices entiers des classes. Dans notre cas, les sorties ont la forme `(N, 2)` et les cibles la forme `(N,)`, avec le type `torch.long`.

**1. Suivre les dimensions.** Pour sept observations de dimension deux, écrivez les formes successives dans l’architecture

$$2\;\longrightarrow\;32\;\longrightarrow\;32\;\longrightarrow\;2.$$

Les deux couches cachées sont suivies d’une tangente hyperbolique `Tanh`. La dernière couche reste affine : **n’ajoutez pas de softmax en sortie**. La transformation en probabilités sera utilisée pour les graphiques, pas avant le calcul de cette perte.

**2. Compléter la construction.** Remplacez les cinq emplacements de `make_mlp`. Employez `width` plutôt que la constante `32` à l’intérieur de la fonction, afin de conserver l’interface proposée. `nn.Sequential` assemble les modules dans l’ordre où ils apparaissent.

**3. Contrôler avant d’entraîner.** Exécutez le passage sur sept exemples. Vérifiez la forme des sorties et comptez les paramètres des trois couches affines, biais compris. Comparez votre somme au nombre affiché. L’activation ajoute-t-elle des paramètres ?


<details>
<summary>Aide : écrire une couche à la fois</summary>

`nn.Linear(d_entree, d_sortie)` construit une transformation affine. Une couche de $d$ entrées et $h$ sorties contient $dh+h$ paramètres, en comptant le biais. `nn.Tanh()` transforme chaque composante sans modifier la forme du tenseur. Pour les trois couches affines, lisez successivement les trois flèches du schéma.

</details>


In [ ]:
def make_mlp(width=32):
    """Construire le classifieur 2 → width → width → 2, à sorties logits."""
    layer_in = None       # Première transformation affine : deux coordonnées en entrée.
    activation_1 = None   # Non-linéarité après la première couche cachée.
    layer_hidden = None   # Deuxième transformation affine, entre les couches cachées.
    activation_2 = None   # Non-linéarité après la deuxième couche cachée.
    layer_out = None      # Deux logits en sortie, sans activation supplémentaire.
    verifier_completion("B1", layer_in=layer_in, activation_1=activation_1,
                        layer_hidden=layer_hidden, activation_2=activation_2, layer_out=layer_out)
    return nn.Sequential(layer_in, activation_1, layer_hidden, activation_2, layer_out)


In [ ]:
probe = make_mlp()
assert probe(X_train[:7]).shape == (7, 2)
assert not any(isinstance(m, nn.Softmax) for m in probe.modules())
print(probe)
print("Paramètres entraînables :", sum(p.numel() for p in probe.parameters()))

assert sum(p.numel() for p in probe.parameters()) == 1218, "Reprenez les dimensions et les biais."
assert sum(isinstance(m, nn.Tanh) for m in probe.modules()) == 2, "Deux activations Tanh sont attendues."
assert sum(isinstance(m, nn.Linear) for m in probe.modules()) == 3, "Trois couches affines sont attendues."


### Outils fournis — À exécuter, sans les réécrire

Le moteur d’entraînement est fourni pour que B porte sur la représentation. Vous écrirez vous-même une boucle plus courte en C. Pour l’instant, lisez son contrat : `train_classifier` reçoit un modèle, l’entraînement et la validation ; elle ne reçoit **aucune donnée de test**.

Elle ajuste les poids avec AdamW sur des mini-lots de 64 exemples, puis recalcule la perte d’entraînement et celle de validation après chaque époque. Elle conserve une copie des poids lorsque la perte de validation s’améliore et restaure cette copie à la fin. Le modèle rendu correspond donc à `best_epoch`, pas nécessairement à la dernière époque exécutée.

Le résultat est un dictionnaire. Vous utiliserez `model` pour les prédictions, `history` pour les courbes, `best_epoch` pour l’époque retenue et `best_val_loss` pour le critère de sélection. Les figures distinguent la perte d’entraînement, la perte de validation et l’exactitude de validation, c’est-à-dire la proportion de décisions correctes.

**Protocole annoncé avant les résultats.** D sélectionnera un candidat parmi quatre : le modèle affine et le MLP de B, entraînés pendant 180 époques, puis les deux MLP de D, entraînés pendant 350 époques avec deux valeurs de décroissance des poids. Le critère sera la meilleure perte de validation. Le modèle de vérification construit en C n’entrera pas dans cette comparaison.


In [ ]:
loss_fn = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate_classifier(model, x, y):
    model.eval()
    logits = model(x)
    return {"loss": float(loss_fn(logits, y)),
            "accuracy": float((logits.argmax(dim=1) == y).float().mean())}

def train_classifier(model, x_train, y_train, x_val, y_val,
                     epochs=180, lr=0.01, weight_decay=0.0, seed=SEED + 2):
    model = model.to(DEVICE)
    loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64,
                        shuffle=True, generator=torch.Generator().manual_seed(seed))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": []}
    best_loss, best_epoch, best_state = float("inf"), 0, None
    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        train_metrics = evaluate_classifier(model, x_train, y_train)
        val_metrics = evaluate_classifier(model, x_val, y_val)
        for split, metrics in [("train", train_metrics), ("val", val_metrics)]:
            for key, value in metrics.items():
                history[f"{split}_{key}"].append(value)
        if val_metrics["loss"] < best_loss:
            best_loss, best_epoch = val_metrics["loss"], epoch
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    model.eval()
    return {"model": model, "history": history, "best_val_loss": best_loss,
            "best_epoch": best_epoch, "epochs": epochs,
            "weight_decay": weight_decay}

def show_histories(experiments, figure_name="classification_learning"):
    fig, axes = plt.subplots(1, 2, figsize=(10.8, 3.5))
    for name, result in experiments.items():
        h = result["history"]
        epochs = np.arange(1, len(h["train_loss"]) + 1)
        line = axes[0].plot(epochs, h["train_loss"], label=name + " train")[0]
        axes[0].plot(epochs, h["val_loss"], "--", color=line.get_color(), label=name + " val")
        axes[1].plot(epochs, h["val_accuracy"], label=name)
    axes[0].set(xlabel="Époque", ylabel="Entropie croisée", title="Perte : apprentissage / validation")
    axes[1].set(xlabel="Époque", ylabel="Exactitude", title="Validation seulement", ylim=(.35, 1.0))
    for ax in axes:
        ax.legend(fontsize=8)
        ax.grid(alpha=.2)
    fig.tight_layout()
    export_figure(fig, figure_name)
    plt.show()

def show_boundaries(experiments):
    grid_axis = torch.linspace(-1.8, 1.8, 130)
    gx, gy = torch.meshgrid(grid_axis, grid_axis, indexing="xy")
    grid_raw = torch.stack((gx.ravel(), gy.ravel()), dim=1)
    grid = (grid_raw - mean_train) / std_train
    fig, axes = plt.subplots(1, len(experiments), figsize=(5.1 * len(experiments), 4.0), squeeze=False)
    for ax, (name, result) in zip(axes[0], experiments.items()):
        model = result["model"]
        model.eval()
        with torch.no_grad():
            probability = model(grid).softmax(dim=1)[:, 1].reshape(gx.shape)
        im = ax.contourf(gx, gy, probability, levels=np.linspace(0, 1, 15), cmap="coolwarm", vmin=0, vmax=1)
        ax.contour(gx, gy, probability, levels=[.5], colors="black", linewidths=1)
        ax.scatter(X_val_raw[:, 0], X_val_raw[:, 1], c=y_val, cmap="coolwarm", s=14, edgecolors="white", linewidths=.35)
        ax.set(title=name + " — points de validation", xlabel="$x_1$", ylabel="$x_2$", aspect="equal")
    fig.colorbar(im, ax=axes.ravel().tolist(), label=r"$p_\theta(y=1\mid x)$", shrink=.8)
    export_figure(fig, "classification_boundaries")
    plt.show()


### B2 — Prévoir une frontière, puis observer celle qui est apprise · 17 min

**Avant le lancement**, dessinez une frontière possible pour chaque famille et répondez à la première ligne de votre trace ci-dessous. Pour le modèle affine, la décision compare deux logits affines : quelle forme peut prendre l’ensemble où ces deux scores sont égaux ? Pour le MLP, que permettent les transformations non linéaires intermédiaires ?

**Lancez ensuite les deux entraînements fournis**, sans modifier leurs réglages. Les deux modèles utilisent les mêmes données, le même budget et la même règle d’optimisation. Ils n’ont pas les mêmes dimensions de paramètres ; la graine commune ne signifie donc pas qu’ils partent des mêmes matrices de poids.

**Lisez les résultats dans cet ordre.** Examinez d’abord la géométrie des frontières. Regardez ensuite si les pertes diminuent et si l’écart entraînement–validation évolue. Enfin, relevez l’époque retenue et l’exactitude de validation. Sur les cartes, le fond représente la probabilité prédite de la classe 1, la courbe de niveau `0.5` est la frontière, et les points appartiennent à la validation.

Décrivez où les modèles se trompent : s’agit-il de régions entières incompatibles avec la frontière choisie, ou de points plus dispersés ? Ne cherchez pas un score prédéterminé. La question est de relier les observations à la capacité de représentation, au bruit d’étiquette et au déroulement de l’apprentissage.


#### Votre trace de travail

**Frontière attendue pour le modèle affine et pour le MLP, avant exécution** — À compléter.


In [ ]:
torch.manual_seed(SEED + 10)
linear = train_classifier(nn.Linear(2, 2), X_train, y_train, X_val, y_val)
torch.manual_seed(SEED + 10)
mlp = train_classifier(make_mlp(), X_train, y_train, X_val, y_val)
experiments_B = {"Linéaire": linear, "MLP": mlp}
for name, result in experiments_B.items():
    metrics = evaluate_classifier(result["model"], X_val, y_val)
    print(f"{name:10s} | meilleure époque {result['best_epoch']:3d} | "
          f"perte val {metrics['loss']:.4f} | exactitude val {metrics['accuracy']:.3f}")
show_histories(experiments_B)
show_boundaries(experiments_B)


### Bilan B — Expliquer le résultat plutôt que commenter un pourcentage · 5 min

Les cinq dernières minutes peuvent être consacrées à une mise en commun. Appuyez votre réponse sur une figure et sur une propriété du modèle.

1. Pourquoi une frontière affine ne peut-elle pas séparer exactement les quatre quadrants alternés, même en prolongeant l’entraînement ?
2. Pourquoi faut-il donner des logits, et non des probabilités déjà transformées, à `CrossEntropyLoss` ?
3. Quels indices vous feraient plutôt suspecter une représentation insuffisante ? Quels contrôles vous feraient plutôt chercher un problème dans l’entraînement ?
4. Un théorème d’approximation universelle garantit-il que le réseau et le budget utilisés ici trouveront une bonne solution à partir de cet échantillon ? Distinguez existence, apprentissage et généralisation.

**Pour passer à C :** formulez une conclusion du type « La figure montre… ; cela est cohérent avec… ; cela ne prouve pas… ».


#### Votre trace de travail

**Observation appuyée sur une figure** — À compléter.

**Interprétation et limite de la conclusion** — À compléter.


<a id="tp-c"></a>
## C — Comprendre ce qui fait apprendre le modèle
**30 minutes · calculer un gradient, le vérifier, puis l’utiliser**

En B, un moteur fourni a modifié les paramètres du réseau. Nous allons maintenant ouvrir ce mécanisme. Il faut distinguer deux opérations : **calculer la sensibilité de la perte aux paramètres**, puis **utiliser cette sensibilité pour déplacer les paramètres**. La première relève de la dérivation ; la seconde, de l’optimiseur.

Nous commencerons par un modèle affine assez petit pour retrouver ses gradients à la main. Nous comparerons ensuite ce calcul à deux contrôles indépendants, avant d’écrire une époque d’entraînement. Gardez les données préparées en A et les fonctions de B : seule la courte boucle demandée en C2 sera à écrire.


### C1 — Retrouver le gradient d’une perte d’entropie croisée · 12 min

**Identifiez d’abord les objets.** Pour ce calcul local, nous notons $n$ le nombre d’exemples, $d$ le nombre de coordonnées et $c$ le nombre de classes. Les exemples sont les lignes de $X$, de forme $(n,d)$ ; les poids $W$ ont la forme $(d,c)$ et le biais $b$ la forme $(c,)$. Les logits sont $Z=XW+b$. Comme en A1, cette convention pour $W$ est la transposée de celle du stockage interne de `nn.Linear`.

La perte est **la moyenne** des entropies croisées sur le lot. Notons $P$ la softmax des lignes de $Z$ et $Y$ la matrice indicatrice des classes : une ligne de $Y$ contient un 1 dans la colonne de la classe attendue et des 0 ailleurs. La dérivée par rapport aux logits est fournie :

$$D_Z=\frac{\partial L}{\partial Z}=\frac{P-Y}{n}.$$

**1. Préparez le calcul sur papier.** Quelle est la forme de $D_Z$ ? Celle du gradient par rapport à $W$ doit être exactement celle de $W$. Quel produit matriciel entre $X$ et $D_Z$ satisfait cette contrainte ? Justifiez ensuite ce produit en considérant l’effet d’une variation d’un poids sur les logits. Pour le biais, demandez-vous combien de lignes reçoivent le même $b$.

**2. Complétez uniquement `dw` et `db`.** Le calcul des probabilités et celui de `dz` sont fournis. N’utilisez ni `backward()` ni `autograd.grad` dans cette fonction : l’objectif est d’exprimer les dérivées à partir des opérations matricielles. Le facteur $1/n$ est déjà inclus dans `dz` ; il ne doit pas être appliqué une seconde fois.

**3. Exécutez le contrôle.** Il compare votre réponse à `autograd`, puis vérifie ce dernier par différences finies centrées avec `gradcheck`. Les cinq exemples artificiels du contrôle comportent **trois classes**, afin de vérifier une fonction générale ; ils ne remplacent pas le jeu XOR à deux classes utilisé ailleurs. Le calcul est effectué en double précision.

**Ce que vous devez observer.** Les formes doivent coïncider et les écarts numériques rester dans les tolérances affichées dans les assertions. Un écart d’un facteur proche de cinq suggère ici une confusion entre somme et moyenne. Un problème de forme invite d’abord à revoir la transposition ou l’axe de sommation.


<details>
<summary>Aide — Remonter des logits vers les poids et le biais</summary>

Pour un poids $W_{jk}$, on a $\partial Z_{ik}/\partial W_{jk}=X_{ij}$. Sa contribution à la perte est donc la somme, sur $i$, de $X_{ij}(D_Z)_{ik}$. Écrivez ce tableau de sommes comme un produit matriciel. Pour $b_k$, la dérivée de chaque $Z_{ik}$ par rapport à $b_k$ vaut 1 : il reste une somme sur les exemples. Avec des exemples en lignes, cet axe est `dim=0`.

</details>


In [ ]:
def manual_ce_grad(x, w, b, y):
    """Calculer les gradients de la perte moyenne, sans différentiation automatique."""
    probabilities = (x @ w + b).softmax(dim=1)
    one_hot = F.one_hot(y, num_classes=w.shape[1]).to(x.dtype)
    dz = (probabilities - one_hot) / len(x)
    dw = None  # Étape 1 : gradient des poids, de même forme que w.
    db = None  # Étape 2 : gradient du biais, de même forme que b.
    verifier_completion("C1", dw=dw, db=db)
    return dw, db


In [ ]:
g = torch.Generator().manual_seed(SEED + 20)
xg = torch.randn(5, 2, generator=g, dtype=torch.float64)
wg = torch.randn(2, 3, generator=g, dtype=torch.float64, requires_grad=True)
bg = torch.randn(3, generator=g, dtype=torch.float64, requires_grad=True)
yg = torch.tensor([0, 1, 2, 0, 1], dtype=torch.long)
lg = F.cross_entropy(xg @ wg + bg, yg)
dw_auto, db_auto = torch.autograd.grad(lg, (wg, bg))
dw_manual, db_manual = manual_ce_grad(xg, wg.detach(), bg.detach(), yg)
assert dw_manual.shape == wg.shape, "C1 — Le gradient des poids doit avoir la forme de wg."
assert db_manual.shape == bg.shape, "C1 — Le gradient du biais doit avoir la forme de bg."
print("Écart maximal sur W :", float((dw_manual - dw_auto).abs().max()))
print("Écart maximal sur b :", float((db_manual - db_auto).abs().max()))
assert torch.allclose(dw_manual, dw_auto, atol=1e-10, rtol=1e-8)
assert torch.allclose(db_manual, db_auto, atol=1e-10, rtol=1e-8)
passed = torch.autograd.gradcheck(lambda w, b: F.cross_entropy(xg @ w + b, yg),
                                 (wg, bg), eps=1e-6, atol=1e-5, rtol=1e-3)
print("Vérification par différences finies :", passed)

assert passed, "C1 — Le contrôle par différences finies doit être satisfait."


### C1 bis — Pourquoi faut-il remettre les gradients à zéro ? · 3 min

Considérons le paramètre scalaire $a=2$ et la perte $L(a)=a^2$. Sa dérivée vaut $2a$. La cellule suivante effectue deux appels à `backward()`, efface ensuite `a.grad`, puis effectue un troisième appel. Aucun optimiseur ne modifie la valeur de $a$.

Avant d’exécuter la cellule, prévoyez les trois nombres imprimés. Après exécution, expliquez la différence éventuelle entre « la dérivée de la perte courante » et « la valeur actuellement stockée dans `.grad` ». Observez que l’expression `a * a` est recalculée avant chaque appel : il ne s’agit pas de redériver trois fois un même graphe déjà libéré.


#### Votre trace de travail

**Trois nombres prévus, puis explication de leur différence** — À compléter.


In [ ]:
a = torch.tensor(2., requires_grad=True)
(a * a).backward()
first = a.grad.item()
(a * a).backward()
accumulated = a.grad.item()
a.grad = None
(a * a).backward()
after_reset = a.grad.item()
print("Premier backward / second sans effacement / après effacement :", first, accumulated, after_reset)
assert (first, accumulated, after_reset) == (4., 8., 4.)


### C2 — Écrire une époque d’entraînement · 10 min

Une **époque** est un parcours du jeu d’entraînement. Le chargeur fournit successivement des mini-lots `(xb, yb)` ; chaque lot doit produire une mise à jour. Vous disposez déjà de `model`, de `optimizer` et de la perte `loss_fn` définie en B. La fonction demandée ne doit ni sélectionner un modèle ni consulter la validation ou le test.

**Reconstituez d’abord la chronologie.** Pour chaque lot, il faut préparer les emplacements de gradients, calculer les logits, construire une perte scalaire, calculer les gradients, puis mettre à jour les paramètres. Quelle opération correspond à chacune de ces étapes ? Laquelle change effectivement les poids ?

**Complétez ensuite les cinq emplacements numérotés.** `model.train()` et le calcul de la moyenne de fin d’époque sont fournis. Les instructions 1, 4 et 5 doivent être ajoutées à la place des commentaires correspondants ; les valeurs `None` de `logits` et de `loss` doivent être remplacées.

**Vérifiez localement, puis entraînez.** Un petit contrôle fourni suit l’ordre des opérations et vérifie la moyenne sur un dernier lot plus court. Exécutez-le avant les 30 époques de l’expérience. L’entraînement de vérification reprend le MLP de B, mais il ne participe pas à la sélection de D.

**Lisez correctement la perte renvoyée.** Chaque perte de lot est une moyenne. On la multiplie par la taille du lot avant de sommer, puis on divise par le nombre total d’exemples. Le dernier lot n’a donc pas artificiellement le même poids qu’un lot complet. Les pertes ont toutefois été calculées avec des paramètres successifs : leur moyenne n’est pas la perte du modèle final réévalué sur tout l’entraînement.


<details>
<summary>Aide — Les opérations utiles et leur rôle</summary>

`optimizer.zero_grad(set_to_none=True)` prépare une nouvelle accumulation. `model(xb)` réalise le passage avant. La perte s’appelle comme une fonction : `loss_fn(logits, yb)`. Une perte scalaire permet l’appel à `.backward()`. Enfin, `optimizer.step()` utilise les gradients disponibles pour modifier les paramètres. Ces appels ont des rôles distincts ; il reste à les placer dans la chronologie de votre boucle.

</details>


In [ ]:
def train_one_epoch(model, loader, optimizer):
    """Effectuer un parcours des lots et renvoyer leur perte moyenne pondérée."""
    model.train()
    total_loss, total_examples = 0.0, 0
    for xb, yb in loader:
        # 1. À compléter : préparer les gradients pour le lot courant.

        logits = None  # 2. À compléter : calculer les logits du lot.
        verifier_completion("C2 — passage avant", logits=logits)
        loss = None    # 3. À compléter : calculer la perte scalaire du lot.
        verifier_completion("C2 — perte", loss=loss)

        # 4. À compléter : calculer les gradients de cette perte.

        # 5. À compléter : effectuer la mise à jour des paramètres.

        # Comptabilisation fournie : tenir compte de la taille du dernier lot.
        total_loss += loss.detach().item() * len(xb)
        total_examples += len(xb)
    return total_loss / total_examples


**Contrôle fourni — À exécuter sans le modifier.** Ce test emploie cinq exemples répartis en lots de tailles 2, 2 et 1. Il observe l’ordre des appels et compare la perte renvoyée à une moyenne par exemple. Une erreur précise le point à reprendre ; le détail de l’instrumentation n’est pas un exercice supplémentaire.


In [ ]:
def controler_une_epoque():
    """Contrôler la chronologie et la pondération, sans changer l'aléa du TP."""
    events, batch_logits = [], []

    class ObservedLinear(nn.Linear):
        def forward(self, x):
            events.append("prédiction")
            output = super().forward(x)
            batch_logits.append(output.detach().clone())
            def observe_gradient(gradient):
                events.append("gradient")
                return gradient
            output.register_hook(observe_gradient)
            return output

    class ObservedSGD(torch.optim.SGD):
        def zero_grad(self, *args, **kwargs):
            events.append("remise à zéro")
            return super().zero_grad(*args, **kwargs)
        def step(self, *args, **kwargs):
            events.append("mise à jour")
            return super().step(*args, **kwargs)

    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(17)
        x = torch.tensor([[0., 0.], [1., 0.], [0., 1.], [1., 1.], [2., -1.]])
        y = torch.tensor([0, 1, 1, 0, 1], dtype=torch.long)
        model = ObservedLinear(2, 2)
        model.eval()  # La fonction doit rétablir le mode d'entraînement.
        before = [p.detach().clone() for p in model.parameters()]
        loader = DataLoader(TensorDataset(x, y), batch_size=2, shuffle=False)
        optimizer = ObservedSGD(model.parameters(), lr=.05)
        observed_loss = train_one_epoch(model, loader, optimizer)
        expected_order = ["remise à zéro", "prédiction", "gradient", "mise à jour"] * 3
        assert events == expected_order, (
            "C2 — Reprenez les cinq étapes : effacer, prédire, calculer la perte, "
            f"dériver, mettre à jour. Appels observés : {events}"
        )
        assert model.training, "C2 — L'époque doit placer le modèle en mode entraînement."
        assert any(not torch.equal(a, b) for a, b in zip(before, model.parameters())), (
            "C2 — Les poids doivent effectivement être modifiés."
        )
        expected_loss = F.cross_entropy(torch.cat(batch_logits), y).item()
        assert math.isclose(observed_loss, expected_loss, rel_tol=1e-6, abs_tol=1e-7), (
            "C2 — La moyenne doit être pondérée par le nombre d'exemples de chaque lot."
        )
    print("C2 — Chronologie, modification des poids et moyenne pondérée : contrôles satisfaits.")

controler_une_epoque()


In [ ]:
torch.manual_seed(SEED + 30)
model_C = make_mlp()
loader_C = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True,
                      generator=torch.Generator().manual_seed(SEED + 31))
optimizer_C = torch.optim.AdamW(model_C.parameters(), lr=.01, weight_decay=0.)
losses_C = [train_one_epoch(model_C, loader_C, optimizer_C) for _ in range(30)]
assert np.isfinite(losses_C).all()
print("Perte moyenne rencontrée dans les mini-lots :", round(losses_C[0], 4), "→", round(losses_C[-1], 4))
print("Validation après 30 époques :", evaluate_classifier(model_C, X_val, y_val))
print("Cette perte de mini-lots agrège des paramètres successifs ; elle diffère d'une perte recalculée en fin d'époque.")


### Bilan C — Relier les instructions au raisonnement · 5 min

Expliquez en une phrase le rôle de `zero_grad()`, de `backward()` et de `step()`. Précisez ensuite pourquoi la moyenne des pertes rencontrées pendant une époque peut différer d’une perte recalculée après cette époque avec le modèle figé.

Terminez par deux distinctions utiles pour la suite : `model.eval()` désactive-t-il le calcul des gradients ? `torch.no_grad()` place-t-il les couches en mode évaluation ? Appuyez-vous sur leur rôle, plutôt que sur leur nom. Vous retrouverez cette distinction en E, où l’on aura besoin de dériver la sortie d’un modèle déjà entraîné.


#### Votre trace de travail

**Rôle des trois opérations et lecture de la perte** — À compléter.

**Différence entre mode évaluation et suivi des gradients** — À compléter.


<a id="tp-d"></a>
## D — Choisir un modèle sans faire répondre le test à sa place
**25 minutes · régularisation, sélection sur validation et évaluation finale**

Vous savez désormais faire diminuer une perte. Reste à décider quelle version du modèle conserver. Une expérience peut continuer à mieux ajuster ses observations d’entraînement alors que sa performance sur la validation se dégrade. Nous allons examiner cette évolution, puis effectuer un choix selon le protocole annoncé en B.

Le jeu de test préparé en A est resté à l’écart. Ne l’utilisez pas pour départager les candidats, choisir une époque ou modifier un réglage. Il servira, une fois le choix arrêté, à évaluer le système sélectionné.

### Expérience préparatoire — Comparer deux décroissances des poids · 10 min

**Avant de lancer**, formulez une hypothèse sur l’effet possible d’un entraînement plus long et d’une décroissance des poids. Une hypothèse utile n’est pas « le modèle régularisé sera meilleur », mais une proposition que les courbes pourraient contredire : par exemple, une différence attendue entre la poursuite de l’ajustement sur l’entraînement et son transfert à la validation.

Les deux nouvelles expériences utilisent le même MLP, les mêmes poids initiaux, le même ordre de mini-lots, AdamW avec un taux de `0.01` et un budget de 350 époques. Seul `weight_decay` change : `0.0` ou `0.02`. Il s’agit ici de la **décroissance découplée des poids d’AdamW**, et non d’un terme quadratique explicitement ajouté à la perte affichée.

**Exécutez la cellule, puis examinez les courbes.** Les deux pertes sont recalculées en mode évaluation après chaque époque. Repérez, pour chaque expérience, l’époque où la validation est la meilleure et ce qui se produit ensuite. Une amélioration de l’entraînement se traduit-elle toujours par une amélioration de la validation ? La décroissance des poids modifie-t-elle ce constat dans cette réalisation ?

La comparaison contrôlée est celle des **deux expériences de D entre elles**. Comparer directement un MLP de B à un MLP de D ne permet pas d’isoler le seul effet de la durée : leur graine d’initialisation diffère également.


#### Votre trace de travail

**Hypothèse avant lancement et observation susceptible de la contredire** — À compléter.


In [ ]:
experiments_D = {}
for wd in (0.0, 0.02):
    torch.manual_seed(SEED + 40)  # même initialisation dans les deux expériences
    result = train_classifier(make_mlp(), X_train, y_train, X_val, y_val,
                              epochs=350, lr=.01, weight_decay=wd, seed=SEED + 41)
    name = f"MLP long, wd={wd:g}"
    experiments_D[name] = result
    print(f"{name:24s} | meilleure époque {result['best_epoch']:3d} | perte val {result['best_val_loss']:.4f}")
show_histories(experiments_D, "regularization_learning")


### D1 — Passer d’une comparaison à un choix explicite · 7 min

Quatre candidats sont maintenant disponibles : les deux modèles de B et les deux modèles de D. Chaque candidat possède une histoire d’apprentissage, mais aussi un état déjà retenu à son époque de meilleure validation. Le moteur a restauré cet état avant de renvoyer le modèle.

**1. Lisez le tableau fourni ci-dessous.** Identifiez à la main le candidat dont `best_val_loss` est la plus petite et notez son époque. Ne choisissez ni la plus petite perte d’entraînement, ni la meilleure exactitude de test : le critère annoncé est la perte de validation.

**2. Complétez `choose_by_validation`.** Cette fonction reçoit le dictionnaire des candidats et renvoie **le nom** du candidat retenu, c’est-à-dire une clé du dictionnaire. Elle n’a pas à réentraîner les modèles, à modifier leurs poids ou à lire le test. En cas d’égalité exacte, l’un des candidats minimisant le critère convient.

**3. Vérifiez le résultat.** Le nom renvoyé doit correspondre à votre lecture du tableau. La cellule prépare `chosen_model` pour l’évaluation finale. À partir de ce point, le choix de configuration et d’époque est arrêté.

**Questions de reprise.** Pourquoi une simple affectation `best_state = model.state_dict()` ne suffit-elle pas toujours à figer les poids d’une époque ? Pourquoi la validation participe-t-elle au choix du modèle alors qu’aucun `backward()` n’est calculé sur ses observations ? Repérez la copie et la comparaison correspondantes dans le moteur fourni, sans le réécrire.


In [ ]:
candidates = {**experiments_B, **experiments_D}
print(f"{'Candidat':27s} | {'Époque retenue':>14s} | {'Perte de validation':>20s}")
print("-" * 71)
for name, result in candidates.items():
    print(f"{name:27s} | {result['best_epoch']:14d} | {result['best_val_loss']:20.6f}")


<details>
<summary>Aide — Sélectionner une clé de dictionnaire selon un critère</summary>

Dans `candidates[nom]`, la quantité recherchée se trouve sous la clé `"best_val_loss"`. La fonction Python `min` peut comparer des éléments selon une fonction passée par l’argument `key`. On souhaite parcourir les noms et leur associer la perte correspondante, sans renvoyer directement cette perte.

</details>


In [ ]:
def choose_by_validation(candidates):
    """Renvoyer le nom du candidat de plus faible perte de validation."""
    chosen_name = None  # À compléter : sélectionner une clé du dictionnaire.
    verifier_completion("D1", chosen_name=chosen_name)
    return chosen_name


In [ ]:
chosen_name = choose_by_validation(candidates)
assert chosen_name in candidates, "D1 — La fonction doit renvoyer un nom de candidat."
chosen_result = candidates[chosen_name]
assert chosen_result["best_val_loss"] == min(r["best_val_loss"] for r in candidates.values()), (
    "D1 — Reprenez le critère : la plus faible perte de validation, et non d'entraînement."
)
chosen_model = chosen_result["model"]
print("Choix arrêté :", chosen_name, "| époque", chosen_result["best_epoch"])


### D2 — Évaluer une fois le choix retenu · 3 min

Avant d’exécuter la cellule, vérifiez que vous pouvez nommer le candidat, son époque et le critère qui l’a sélectionné. La cellule applique au test **la transformation apprise en A sur l’entraînement**, puis calcule la perte, l’exactitude et un intervalle de Wilson à 95 % pour la proportion de décisions correctes. Le calcul de cet intervalle est fourni ; vous n’avez pas à l’implémenter.

Exécutez la cellule une fois, puis relevez le résultat. L’intervalle rappelle qu’une proportion mesurée sur 180 observations n’est pas une constante connue sans incertitude. Il ne décrit pas toute la variabilité d’un réentraînement et ne prouve pas que le même score vaudrait sur une autre population.

La cellule garde le premier rapport affiché lors de son exécution dans le noyau. Ce garde-fou matérialise une règle de méthode ; il n’empêche pas techniquement de relire les données. **Ne revenez pas modifier les réglages à la lumière de ce score** : le test servirait alors à la sélection et perdrait son rôle d’évaluation réservée.


In [ ]:
def wilson_interval(k, n, z=1.959963984540054):
    p = k / n
    denominator = 1 + z*z/n
    center = (p + z*z/(2*n)) / denominator
    radius = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / denominator
    return center - radius, center + radius

if globals().get("TEST_ALREADY_OPENED", False):
    print("Test déjà ouvert : conserver le résultat initial ci-dessous, sans nouveau réglage.")
    print(final_report)
else:
    X_test_raw, y_test = TEST_SCELLE
    X_test = (X_test_raw - mean_train) / std_train
    chosen_model.eval()
    with torch.no_grad():
        test_logits = chosen_model(X_test)
        test_loss = float(loss_fn(test_logits, y_test))
        correct = int((test_logits.argmax(1) == y_test).sum())
    low, high = wilson_interval(correct, len(y_test))
    final_report = {"model": chosen_name, "best_epoch": chosen_result["best_epoch"],
                    "test_loss": test_loss, "test_accuracy": correct / len(y_test),
                    "wilson_95": (low, high), "n_test": len(y_test)}
    TEST_ALREADY_OPENED = True
    print(final_report)


### Bilan D — Séparer le choix, le résultat et sa portée · 5 min

Rédigez une conclusion en trois temps : indiquez le critère et l’état retenus ; rapportez la performance mesurée sur le test ; précisez la population et les limites auxquelles cette mesure se rapporte. Une phrase par point suffit.

Pour préparer la discussion, examinez également ces questions :

1. Repérez sur les courbes l’époque qui minimise la perte d’entraînement, puis comparez-la à l’époque retenue. Les deux règles répondent-elles à la même question ?
2. Que se passe-t-il si vous utilisez maintenant le score de test pour choisir un autre taux d’apprentissage ?
3. Deux configurations peuvent-elles avoir une exactitude identique et des pertes différentes ?
4. Une amélioration observée avec `weight_decay=0.02` suffit-elle à établir la supériorité générale de ce réglage ? L’intervalle d’exactitude d’un seul modèle permet-il de conclure à cette supériorité ?


#### Votre trace de travail

**Choix effectué et critère** — À compléter.

**Résultat de test avec son intervalle** — À compléter.

**Portée de la conclusion** — À compléter.


<a id="tp-e"></a>
## E — Apprendre une trajectoire à partir d’une équation
**30 minutes · un oscillateur amorti comme problème de contrôle**

Jusqu’ici, la cible indiquait la classe attendue pour chaque entrée. Considérons maintenant une situation différente : nous connaissons une loi d’évolution et des conditions initiales, mais nous ne fournissons pas au réseau les valeurs de la trajectoire à apprendre. Nous pouvons pourtant calculer son défaut de satisfaction de la loi, puis utiliser ce défaut comme perte. C’est le principe du réseau informé par la physique étudié ici, ou **PINN**.

Le problème est un oscillateur amorti :

$$x''(t)+2\gamma x'(t)+\omega_0^2x(t)=0,
\qquad x(0)=x_0=1,\qquad x'(0)=v_0=0.$$

Les coefficients sont connus : $\gamma=0{,}3\ \mathrm{s}^{-1}$, $\omega_0=2\ \mathrm{s}^{-1}$, et nous étudions $0\leq t\leq T=3\ \mathrm{s}$. Le déplacement est normalisé par son amplitude initiale. Nous choisissons ce problème parce qu’une solution analytique et un intégrateur classique permettront de contrôler le résultat : leur rôle n’est pas de fournir les cibles de l’entraînement.

**Conventions à garder sous les yeux.** Le réseau reçoit le temps sans dimension $s=t/T$, et sa sortie est notée $u_\theta(s)=x_\theta(Ts)$. La même sortie en temps sans dimension est notée $v_\theta(s)$ ; nous conservons ici les noms `u`, `du` et `d2u` du notebook initial. Dans le code, `du` et `d2u` désignent donc des dérivées **par rapport à $s$**, pas directement une vitesse et une accélération physiques.

Le modèle fourni impose les conditions initiales par sa construction :

$$u_\theta(s)=x_0+T v_0s+s^2N_\theta(s),$$

où $N_\theta$ est un MLP à activations `Tanh`. Avant de l’entraîner, substituez $s=0$, puis dérivez cette expression une fois. Vous devez retrouver les deux conditions initiales, quelle que soit la valeur des poids du réseau.


### Préparation fournie — Le modèle et les lieux où vérifier l’équation

Exécutez la cellule suivante. Elle construit le réseau en double précision et 80 **points de collocation** sur $[0,1]$. Ces points sont les positions où nous demanderons au réseau de satisfaire l’équation ; ils ne portent aucune valeur mesurée de la trajectoire.

La fonction `exact_solution` est fournie pour deux contrôles : vérifier que notre code du résidu reconnaît une solution exacte, puis évaluer la trajectoire obtenue après l’entraînement. Elle ne sera appelée ni pour produire une cible d’apprentissage ni dans la perte optimisée par le PINN.

Dans le régime sous-amorti, avec $\omega_d=\sqrt{\omega_0^2-\gamma^2}$, cette référence s’écrit

$$x_\star(t)=e^{-\gamma t}\left[x_0\cos(\omega_dt)+\frac{v_0+\gamma x_0}{\omega_d}\sin(\omega_dt)\right].$$

Le terme en sinus permet de respecter la vitesse initiale. Il est déjà présent dans le code fourni ; aucune dérivation complète de cette solution n’est demandée ici.


In [ ]:
GAMMA, OMEGA0, T_FINAL = 0.3, 2.0, 3.0
X0, V0 = 1.0, 0.0
PINN_DTYPE = torch.float64

def exact_solution(t):
    omega_d = math.sqrt(OMEGA0**2 - GAMMA**2)
    return torch.exp(-GAMMA * t) * (X0 * torch.cos(omega_d * t)
           + (V0 + GAMMA * X0) / omega_d * torch.sin(omega_d * t))

class OscillatorPINN(nn.Module):
    def __init__(self, width=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, width), nn.Tanh(),
                                 nn.Linear(width, width), nn.Tanh(), nn.Linear(width, 1))
    def forward(self, s):
        return X0 + T_FINAL * V0 * s + s.square() * self.net(s)

torch.manual_seed(SEED + 50)
pinn = OscillatorPINN().to(dtype=PINN_DTYPE, device=DEVICE)
s_collocation = torch.linspace(0., 1., 80, dtype=PINN_DTYPE).reshape(-1, 1)
print("Paramètres du PINN :", sum(p.numel() for p in pinn.parameters()))
print("Nombre de points de collocation :", len(s_collocation))


### E1 — Des dérivées du réseau au résidu physique · 10 min, préparation comprise

**1. Retrouvez les facteurs de changement de variable.** Puisque $s=t/T$, la règle de chaîne donne $x'_\theta(t)=u'_\theta(s)/T$ et $x''_\theta(t)=u''_\theta(s)/T^2$. Écrivez l’équation avec ces dérivées, puis divisez-la par $\omega_0^2$. Le résidu normalisé à calculer est

$$\widetilde r_\theta(s)=
\frac{u''_\theta(s)}{\omega_0^2T^2}
+\frac{2\gamma\,u'_\theta(s)}{\omega_0^2T}
+u_\theta(s).$$

**2. Complétez la fonction dans cet ordre : `du`, puis `d2u`, puis `residual`.** La copie de l’entrée avec `requires_grad_(True)` et le passage avant sont fournis. Le premier appel à `torch.autograd.grad` doit dériver `u` par rapport à `s`, le second doit dériver `du` par rapport à `s`. Les tenseurs ont tous la forme `(nombre_de_points, 1)`.

Pour chaque appel, utilisez un tenseur de 1 de la même forme que la sortie dans `grad_outputs`, et conservez le graphe avec `create_graph=True`. La première dérivée doit pouvoir être dérivée à nouveau ; la seconde entrera dans une perte qu’il faudra ensuite dériver par rapport aux poids. Cette distinction entre dérivation en $s$ et dérivation en $\theta$ est au cœur de l’exercice.

**3. Exécutez les contrôles avant d’entraîner.** Un premier test, sur une fonction polynomiale simple, vérifie les deux dérivées. Le second applique votre résidu à la solution analytique connue et vérifie les conditions initiales du réseau non entraîné. Si le résidu de la référence n’est pas proche de zéro, revoyez les facteurs de $T$ et de $\omega_0$ avant de mettre en cause l’optimiseur.

Ne placez pas ces calculs dans un contexte `torch.no_grad()` : même lorsque les poids ne changent plus, le résidu a besoin des dérivées par rapport à l’entrée.

**Pourquoi `Tanh` ?** Le résidu demande une dérivée seconde régulière. Un MLP ReLU ordinaire est affine par morceaux et possède des ruptures de dérivée ; le facteur $s^2$ de notre construction ne supprime pas en général ces ruptures. L’activation lisse fournie permet ici de poser le résidu fort sans cette difficulté.


<details>
<summary>Aide — Lire un appel à torch.autograd.grad</summary>

Le schéma utile est `torch.autograd.grad(sortie, entree, grad_outputs=torch.ones_like(sortie), create_graph=True)[0]`. Le résultat est un tuple ; `[0]` en extrait la dérivée relative à l’unique entrée demandée. Remplacez `sortie` et `entree` pour chacun des deux appels.

L’argument `grad_outputs` demande ici la dérivée de la somme des sorties. Elle donne les dérivées point par point parce que le MLP traite chaque ligne indépendamment : la sortie en $s_i$ ne dépend pas des autres $s_j$. Une opération mélangeant les points demanderait un autre examen de la jacobienne.

</details>


In [ ]:
def derivatives_and_residual(model, s_values):
    """Renvoyer la sortie, ses deux dérivées en s et le résidu normalisé."""
    s = s_values.detach().clone().requires_grad_(True)
    u = model(s)
    du = None  # Étape 1 : dériver u par rapport à s, en conservant le graphe.
    verifier_completion("E1 — première dérivée", du=du)
    d2u = None # Étape 2 : dériver du par rapport à s, en conservant le graphe.
    verifier_completion("E1 — seconde dérivée", d2u=d2u)
    residual = None  # Étape 3 : assembler le résidu avec les facteurs de changement de variable.
    verifier_completion("E1 — résidu", residual=residual)
    return u, du, d2u, residual


In [ ]:
# Un contrôle local des dérivées, avant le contrôle de l'équation physique.
class PolynomialReference(nn.Module):
    def forward(self, s):
        return 1 + 2*s + 3*s.square()

s_probe = torch.tensor([[0.], [.2], [.7]], dtype=PINN_DTYPE)
u_probe, du_probe, d2u_probe, _ = derivatives_and_residual(PolynomialReference(), s_probe)
assert u_probe.shape == du_probe.shape == d2u_probe.shape == s_probe.shape, (
    "E1 — Sortie et dérivées doivent conserver la forme (nombre de points, 1)."
)
assert torch.allclose(du_probe, 2 + 6*s_probe, atol=1e-12, rtol=1e-12), (
    "E1 — La première dérivée de 1 + 2s + 3s² vaut 2 + 6s."
)
assert torch.allclose(d2u_probe, torch.full_like(s_probe, 6), atol=1e-12, rtol=1e-12), (
    "E1 — La seconde dérivée du polynôme doit valoir 6."
)
print("E1 — Les deux dérivées du polynôme de contrôle sont correctes.")


In [ ]:
class ExactReference(nn.Module):
    def forward(self, s):
        return exact_solution(T_FINAL * s)

_, _, _, exact_residual = derivatives_and_residual(ExactReference(), s_collocation)
assert float(exact_residual.detach().abs().max()) < 1e-10
u0, du0, _, _ = derivatives_and_residual(pinn, torch.zeros(1, 1, dtype=PINN_DTYPE))
assert abs(float(u0.detach()) - X0) < 1e-12
assert abs(float(du0.detach()) / T_FINAL - V0) < 1e-12
print("Résidu normalisé de la solution exacte :", float(exact_residual.detach().abs().max()))
print("Conditions initiales imposées avant entraînement :", float(u0.detach()), float(du0.detach()) / T_FINAL)


### E2 — Entraîner sans fournir la trajectoire comme cible · 8 min

Le résidu est maintenant calculable. La perte sera simplement sa moyenne quadratique sur les points de collocation :

$$L_{\mathrm{phys}}(\theta)=\frac{1}{80}\sum_{j=1}^{80}\widetilde r_\theta(s_j)^2.$$

**Avant l’exécution, lisez les quatre lignes centrales de la boucle fournie.** Retrouvez le calcul du résidu, la construction de la perte, la rétropropagation et la mise à jour. Vérifiez qu’aucune valeur de `exact_solution` n’intervient. Où l’information sur la trajectoire entre-t-elle alors dans le calcul ? Distinguez les coefficients de l’équation, les conditions initiales et les points où l’on contrôle la loi.

**Lancez l’entraînement sans changer ses réglages.** Le code effectue 1 200 pas d’Adam, puis un affinage L-BFGS. Ce second optimiseur et sa fonction `closure` sont fournis : leur implémentation n’est pas demandée dans cette première séance. La fonction `closure` recalcule la perte et ses gradients à la demande de l’optimiseur ; elle peut être appelée plusieurs fois pour un seul pas. Son nombre d’appels n’est donc pas un nombre d’époques.

**Pendant le calcul, préparez le contrôle suivant.** Une petite perte sur 80 points garantit-elle que la trajectoire est correcte entre ces points ? Quelle information indépendante vous manque encore ? Ne réentraînez pas plusieurs fois la même instance pour tenter d’atteindre une valeur attendue : une nouvelle expérience doit repartir d’un modèle neuf et annoncer son budget.


In [ ]:
def fit_pinn(model, points, adam_steps=1200, lbfgs_iterations=200):
    trace = []
    adam = torch.optim.Adam(model.parameters(), lr=0.002)
    model.train()
    for step in range(adam_steps):
        adam.zero_grad(set_to_none=True)
        _, _, _, residual = derivatives_and_residual(model, points)
        loss = residual.square().mean()
        loss.backward()
        adam.step()
        trace.append(float(loss.detach()))
    lbfgs = torch.optim.LBFGS(model.parameters(), lr=1.0,
                             max_iter=lbfgs_iterations, max_eval=350,
                             tolerance_grad=1e-10, tolerance_change=1e-12,
                             line_search_fn="strong_wolfe")
    closure_values = []
    def closure():
        lbfgs.zero_grad(set_to_none=True)
        _, _, _, residual = derivatives_and_residual(model, points)
        loss = residual.square().mean()
        loss.backward()
        closure_values.append(float(loss.detach()))
        return loss
    lbfgs.step(closure)
    model.eval()
    return trace, closure_values


In [ ]:
pinn_start = time.perf_counter()
pinn_trace, lbfgs_trace = fit_pinn(pinn, s_collocation)
pinn_seconds = time.perf_counter() - pinn_start
print(f"Entraînement PINN : {pinn_seconds:.1f} s sur cet environnement")
print("Évaluations Adam / appels de la fonction L-BFGS :", len(pinn_trace), len(lbfgs_trace))
print("Dernière perte enregistrée Adam / L-BFGS :", pinn_trace[-1], lbfgs_trace[-1])
# Recalcul de la perte pour l'état effectivement rendu, hors recherche de pas L-BFGS.
_, _, _, final_collocation_residual = derivatives_and_residual(pinn, s_collocation)
final_collocation_loss = float(final_collocation_residual.detach().square().mean())
print("Perte de collocation de l'état final :", final_collocation_loss)


### E3 — Contrôler la solution ailleurs et conclure · 12 min

Nous allons maintenant distinguer trois objets : la **perte optimisée sur les points de collocation**, le **résidu sur une grille indépendante** et l’**erreur de trajectoire par rapport à une référence**. Ils ne mesurent pas la même chose.

**1. Exécutez le contrôle fourni sur 400 nouveaux points.** La grille est distincte de celle d’entraînement, ce que vérifie une assertion. Le code calcule l’erreur relative discrète $\|x_\theta-x_\star\|_2/\|x_\star\|_2$, l’erreur absolue maximale et la racine de la moyenne des carrés du résidu physique. Il contrôle aussi la position et la vitesse initiales. Repérez à chaque fois si une quantité est normalisée ou exprimée en unités physiques.

**2. Regardez les trois graphiques dans l’ordre.** La trajectoire se superpose-t-elle visuellement à la référence ? Le résidu reste-t-il uniforme ou présente-t-il des régions plus défavorables ? La perte de collocation diminue-t-elle, et comment évolue-t-elle entre les deux phases d’optimisation ? Appuyez le diagnostic visuel sur les nombres imprimés : des courbes presque superposées peuvent masquer une erreur mesurable.

**3. Exécutez la référence `solve_ivp`.** Cet intégrateur résout la même équation avec les mêmes conditions initiales. Comparez sa solution à la formule analytique et son temps de calcul à celui de l’entraînement du PINN. Il ne s’agit pas de démontrer un avantage de vitesse du réseau : le problème sert ici à comprendre et à contrôler une méthode.

**4. Rédigez votre bilan.** Expliquez pourquoi les conditions initiales doivent accompagner l’équation, pourquoi les facteurs de $T$ sont indispensables, et pourquoi `create_graph=True` apparaît dans les deux appels de dérivation. Concluez avec une mesure d’erreur, une mesure de résidu et une limite de l’expérience. Ces contrôles portent sur des points finis ; ils ne sont pas une preuve d’exactitude à tout instant.


In [ ]:
s_check = ((torch.arange(400, dtype=PINN_DTYPE) + .5) / 400).reshape(-1, 1)
assert torch.cdist(s_check, s_collocation).min() > 1e-8
# Pas de no_grad ici : les dérivées par rapport à l'entrée sont nécessaires.
u_check, _, _, normalized_residual = derivatives_and_residual(pinn, s_check)
t_check = T_FINAL * s_check
reference = exact_solution(t_check)
relative_l2 = float((torch.linalg.vector_norm(u_check - reference)
                     / torch.linalg.vector_norm(reference)).detach())
max_error = float((u_check - reference).detach().abs().max())
residual_rms = float((OMEGA0**2 * normalized_residual.detach()).square().mean().sqrt())
u0, du0, _, _ = derivatives_and_residual(pinn, torch.zeros(1, 1, dtype=PINN_DTYPE))
pinn_report = {"relative_l2": relative_l2, "max_error": max_error,
               "physical_residual_rms": residual_rms,
               "initial_position_error": abs(float(u0.detach()) - X0),
               "initial_velocity_error": abs(float(du0.detach()) / T_FINAL - V0)}
print(pinn_report)
assert all(np.isfinite(v) for v in pinn_report.values())

fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
axes[0].plot(t_check.numpy(), reference.numpy(), color="black", label="Solution analytique")
axes[0].plot(t_check.numpy(), u_check.detach().numpy(), "--", label="PINN")
axes[0].set(xlabel="Temps (s)", ylabel="Déplacement", title="Solution hors collocation")
axes[0].legend(fontsize=8)
axes[1].plot(t_check.numpy(), (OMEGA0**2 * normalized_residual).detach().numpy())
axes[1].set(xlabel="Temps (s)", ylabel="Résidu physique", title="Équation contrôlée ailleurs")
axes[2].semilogy(np.arange(1, len(pinn_trace)+1), pinn_trace, label="Adam")
axes[2].semilogy(np.arange(len(pinn_trace)+1, len(pinn_trace)+len(lbfgs_trace)+1), lbfgs_trace, label="L-BFGS")
axes[2].set(xlabel="Évaluation de perte", ylabel="Moyenne du résidu normalisé²", title="Optimisation")
axes[2].legend(fontsize=8)
fig.tight_layout()
export_figure(fig, "pinn_controls")
plt.show()


In [ ]:
# Référence numérique classique : contrôle du PINN et de la formule analytique.
from scipy.integrate import solve_ivp
solver_start = time.perf_counter()
t_numpy = t_check[:, 0].numpy()
sol_ivp = solve_ivp(lambda t, z: [z[1], -2*GAMMA*z[1] - OMEGA0**2*z[0]],
                    (0., T_FINAL), [X0, V0], t_eval=t_numpy,
                    method="DOP853", rtol=1e-10, atol=1e-12)
assert sol_ivp.success
ivp_seconds = time.perf_counter() - solver_start
ivp_error = np.max(np.abs(sol_ivp.y[0] - reference[:, 0].numpy()))
print(f"solve_ivp : {ivp_seconds:.4f} s ; écart maximal à la formule analytique : {ivp_error:.3e}")
print("Ces chronométrages illustrent le cas présent ; ils ne constituent pas un benchmark général.")


#### Votre trace de travail

**Information fournie au réseau et rôle des conditions initiales** — À compléter.

**Dérivées, unités et graphe de calcul** — À compléter.

**Erreur, résidu, temps de calcul et portée de la conclusion** — À compléter.


<a id="sortie"></a>
## Ticket de sortie — Retrouver le fil de la séance · 5 min

Fermez les aides et ne lancez pas de nouveau calcul. Répondez en une phrase par question, puis comparez vos réponses avec celles d’un autre binôme. Le but est de reformuler une chaîne de décisions, pas de réciter des commandes.

1. La perte d’entraînement est faible, mais celle de validation reste forte. Proposez deux hypothèses de nature différente et un contrôle permettant de commencer à les distinguer.
2. Des logits de forme `(64, 2)` et des cibles de forme `(64,)` sont-ils compatibles avec l’entropie croisée ? Précisez le type et les valeurs attendus pour les cibles.
3. Un appel à `backward()` modifie-t-il directement les poids ? Quelle instruction le fait ?
4. Pourquoi choisir un hyperparamètre à l’aide du test empêche-t-il de présenter ensuite ce même test comme une évaluation indépendante ?
5. Dans le PINN, quels rôles distincts jouent l’équation et les conditions initiales ?
6. Que vous a appris la représentation du quartet d’Anscombe que son seul tableau de statistiques ne pouvait montrer ?

**Avant de sauvegarder**, vérifiez que vous pouvez expliquer une figure d’Anscombe, une frontière de décision, votre rapport de sélection et de test, ainsi que les contrôles du PINN. Ces résultats restent dans le notebook ; aucune mise en page de rapport supplémentaire n’est demandée ici.


#### Votre trace de travail

**Question 1** — À compléter.

**Question 2** — À compléter.

**Question 3** — À compléter.

**Question 4** — À compléter.

**Question 5** — À compléter.

**Question 6** — À compléter.


## Prolongements facultatifs — Transformer une manipulation en étude

Ces cinq pistes conservent les problèmes ouverts du notebook initial. Elles sont destinées à un travail ultérieur, **pas à être ajoutées aux cinq heures**. Pour chaque étude, formulez une question, gardez une référence et définissez une mesure avant de modifier le code.

### 1. Mesurer la variabilité d’une comparaison

Reprenez les configurations comparées en D, avec un budget fixé à l’avance. Répétez l’expérience sur cinq graines et gardez une comparaison appariée : pour une même graine, les deux réglages de décroissance doivent partir des mêmes poids et parcourir les mêmes lots. Précisez ce que la graine fait varier : l’initialisation seule, ou également la simulation et la partition des observations. Ces deux protocoles ne mesurent pas la même variabilité.

Présentez les pertes de validation par graine, puis leur moyenne et leur dispersion. Demandez-vous si le classement reste stable. Ne réutilisez pas le test déjà ouvert pour poursuivre les réglages : l’objectif de cette étude est de caractériser la variabilité de la comparaison sur validation.

### 2. Distinguer le mode des couches du suivi des gradients

Ajoutez un dropout entre les couches cachées d’une nouvelle instance du MLP. Gardez les mêmes poids et le même petit lot d’entrée pour les comparaisons. Avant de lancer, prévoyez le comportement de quatre cas : `train` ou `eval`, chacun avec ou sans enregistrement des gradients.

Dans chaque cas, effectuez plusieurs prédictions sans réinitialiser la graine entre les appels. Comparez leurs valeurs et examinez `requires_grad` sur les sorties. Décrivez séparément la variabilité des prédictions et la possibilité de les dériver. La dispersion due au dropout n’est pas, à elle seule, une incertitude prédictive calibrée.

### 3. Dégrader le PINN de manière contrôlée

Choisissez **une seule modification** : allonger l’horizon, raréfier la collocation ou remplacer `Tanh` par ReLU. Gardez le cas initial comme référence et repartez chaque fois d’une nouvelle instance. Pour l’horizon, reportez correctement la nouvelle valeur de $T$ dans le changement de variable et dans le résidu.

Comparez la perte de collocation, le résidu physique sur une grille indépendante et l’erreur de trajectoire. Recherchez un cas où ces diagnostics ne racontent pas la même histoire. Pour ReLU, tenez compte du facteur $s^2$ : il peut apporter de la courbure, mais ne supprime pas en général les ruptures de dérivée du réseau. Ne confondez pas les dérivées calculées à l’intérieur des morceaux avec une vérification de l’équation sur les ruptures.

### 4. Passer à un problème inverse

Rendez maintenant **$\gamma$ et $\omega_0$ inconnus**, tout en conservant la forme de l’équation. Produisez des mesures bruitées de la position à quelques instants choisis, puis construisez une perte combinant l’écart aux mesures et le résidu physique. La génération des mesures peut utiliser les paramètres vrais ; le modèle d’estimation ne doit pas y accéder directement.

Une paramétrisation par `softplus` permet d’imposer la positivité. Avant d’entraîner, examinez cependant si les instants observés permettent de distinguer les effets de l’amortissement et de la fréquence. Comparez plusieurs initialisations et niveaux de bruit, puis rapportez séparément l’erreur de trajectoire et l’erreur sur les coefficients. Une bonne courbe ne suffit pas à démontrer l’identification de chaque paramètre.

### 5. Mesurer l’influence d’un point dans le quartet d’Anscombe

Pour chaque jeu, ajustez d’abord la droite complète. Retirez ensuite successivement chacun des onze points et refaites l’ajustement. Enregistrez le changement de pente et d’ordonnée à l’origine, puis représentez-le en fonction du point retiré.

Comparez particulièrement les jeux III et IV : l’un contient un point atypique en ordonnée, l’autre un point de fort levier en abscisse. Que se passe-t-il si le point retiré laisse toutes les abscisses restantes identiques ? Prévoyez ce cas dans votre code plutôt que de laisser une division par zéro produire un résultat trompeur. Reliez ces observations aux résidus et à la robustesse d’une régression.


<a id="optimisateurs"></a>
## Complément expérimental — SGD, RMSprop, Adam et AdamW
**Hors séance · prolongement du chapitre 4**

La partie C a séparé le calcul du gradient de son utilisation. Ce complément examine la seconde opération : pourquoi un même gradient peut-il conduire à des déplacements différents ? Nous isolerons successivement le démarrage des moyennes d’Adam, l’effet de la géométrie et le rôle de la décroissance des poids. Les cellules de calcul sont fournies ; le travail porte sur la prévision et l’interprétation.

### O1 — Comprendre la correction du démarrage d’Adam

Considérez un seul paramètre, un premier gradient $g_1=2$ et des mémoires initialement nulles. Les coefficients sont $\beta_1=0{,}9$, $\beta_2=0{,}999$ et le taux d’apprentissage est $\eta=0{,}1$. Nous notons ici le taux $\eta$ pour ne pas le confondre avec le coefficient `alpha` de RMSprop.

**Sur papier**, calculez successivement $m_1=(1-\beta_1)g_1$, $v_1=(1-\beta_2)g_1^2$, puis $\widehat m_1=m_1/(1-\beta_1)$ et $\widehat v_1=v_1/(1-\beta_2)$. Comparez les quantités positives qui seront soustraites au paramètre, $\eta m_1/\sqrt{v_1}$ et $\eta\widehat m_1/\sqrt{\widehat v_1}$, en négligeant d’abord $\varepsilon$.

**Exécutez ensuite le contrôle numérique**, qui réintroduit $\varepsilon=10^{-8}$. Le déplacement effectif du paramètre est l’opposé de la quantité imprimée. Expliquez pourquoi l’absence de correction peut modifier l’amplitude du premier pas alors que le gradient est inchangé. Cette correction répond-elle à la même question qu’une décroissance des poids ?


In [ ]:
g1 = torch.tensor(2., dtype=torch.float64)
beta1, beta2, eta, eps = .9, .999, .1, 1e-8
m1, v1 = (1-beta1)*g1, (1-beta2)*g1.square()
m1_hat, v1_hat = m1/(1-beta1), v1/(1-beta2)
uncorrected_step = eta * m1 / (v1.sqrt() + eps)
corrected_step = eta * m1_hat / (v1_hat.sqrt() + eps)
print(f"m1={m1:.4f}, v1={v1:.4f}, m1 corrigé={m1_hat:.4f}, v1 corrigé={v1_hat:.4f}")
print(f"Quantité soustraite : sans correction {uncorrected_step:.6f} ; Adam corrigé {corrected_step:.6f}")


#### Votre trace de travail

**Mémoires, quantités soustraites et rôle de la correction** — À compléter.


### O2 — Lire une trajectoire dans une vallée quadratique

Nous minimisons $f(\theta)=\tfrac12\theta^\top H\theta$, avec des valeurs propres de $H$ égales à 1 et 50. Une rotation de 30° rend les directions principales obliques aux axes des paramètres. Tous les algorithmes partent de $(3,3)$ et disposent de 250 mises à jour.

**Avant l’expérience**, rappelez pourquoi la plus forte courbure limite le pas de SGD. Ici, $2/\lambda_{\max}=0{,}04$ et le pas choisi est `0.03`. RMSprop utilise `0.07` avec `alpha=0.9` et sans momentum ; Adam et AdamW utilisent `0.08` avec les coefficients `(0.9, 0.999)`. AdamW ajoute une décroissance de `0.03`. Ces réglages sont ceux du notebook initial : des taux numériques différents et une décroissance présente dans un seul cas empêchent d’y voir une comparaison de performances optimales à protocole identique.

**Exécutez la cellule**, puis reliez chaque courbe de perte à sa trajectoire dans le plan. Observez la rapidité de progression, les changements de direction et les éventuelles oscillations finales. La grandeur affichée est toujours la même fonction quadratique, sans ajout d’un terme de pénalisation.

**Interprétez.** Que peut corriger une adaptation diagonale, et que ne peut-elle pas représenter lorsqu’une vallée est tournée ? Pourquoi une diminution progressive du taux pourrait-elle modifier les oscillations ? Enfin, cette expérience sans jeu de données d’entraînement ni test permet-elle de conclure sur la généralisation d’un réseau ?


In [ ]:
angle = math.pi / 6
rotation = torch.tensor([[math.cos(angle), -math.sin(angle)],
                          [math.sin(angle), math.cos(angle)]], dtype=torch.float64)
H = rotation @ torch.diag(torch.tensor([1., 50.], dtype=torch.float64)) @ rotation.T

def quadratic(theta):
    return .5 * theta @ H @ theta

optimizer_factories = {
    "SGD, lr=.03": lambda params: torch.optim.SGD(params, lr=.03),
    "RMSprop, lr=.07": lambda params: torch.optim.RMSprop(params, lr=.07, alpha=.9, eps=1e-8, momentum=0.),
    "Adam, lr=.08": lambda params: torch.optim.Adam(params, lr=.08, betas=(.9, .999), eps=1e-8),
    "AdamW, lr=.08, wd=.03": lambda params: torch.optim.AdamW(params, lr=.08, betas=(.9, .999), eps=1e-8, weight_decay=.03),
}
optimizer_results = {}
for name, factory in optimizer_factories.items():
    theta = nn.Parameter(torch.tensor([3., 3.], dtype=torch.float64))
    optimizer = factory([theta])
    trajectory = [theta.detach().clone()]
    values = [float(quadratic(theta).detach())]
    for step in range(250):
        optimizer.zero_grad(set_to_none=True)
        value = quadratic(theta)
        value.backward()
        optimizer.step()
        trajectory.append(theta.detach().clone())
        values.append(float(quadratic(theta).detach()))
    optimizer_results[name] = {"trajectory": torch.stack(trajectory).numpy(), "loss": np.array(values)}
    print(f"{name:28s} | perte finale {values[-1]:.6g}")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))
grid = np.linspace(-1.5, 4., 220)
qx, qy = np.meshgrid(grid, grid)
coords = np.stack([qx, qy], axis=-1)
qvalues = .5 * np.einsum("...i,ij,...j->...", coords, H.numpy(), coords)
axes[0].contour(qx, qy, qvalues, levels=[.1, 1, 5, 20, 80, 200], colors="0.8", linewidths=.8)
for name, result in optimizer_results.items():
    trajectory = result["trajectory"]
    axes[0].plot(trajectory[:, 0], trajectory[:, 1], label=name, linewidth=1.3)
    axes[1].semilogy(result["loss"], label=name)
axes[0].scatter([0], [0], marker="*", color="black", s=80)
axes[0].set(xlabel=r"$\theta_1$", ylabel=r"$\theta_2$", title="Trajectoires : vallée quadratique", aspect="equal")
axes[1].set(xlabel="Pas d'optimisation", ylabel="Perte de données", title="Même point initial, 250 pas")
axes[1].legend(fontsize=7)
axes[1].grid(alpha=.2)
fig.tight_layout()
export_figure(fig, "optimizer_trajectories")
plt.show()


#### Votre trace de travail

**Observation des trajectoires** — À compléter.

**Ce que cette expérience établit, et ce qu’elle n’établit pas** — À compléter.


### O3 — Distinguer pénalisation quadratique et décroissance découplée

Nous partons cette fois de $\theta_0=(1,2)$ et d’un gradient de données fixé $g=(-0{,}05,0{,}5)$. Le taux vaut $\eta=0{,}1$ et le coefficient de régularisation $\lambda=0{,}1$. Le code injecte ce gradient connu pour isoler **un seul pas** ; il ne lance pas un nouvel entraînement sur XOR.

**1. Calculez le gradient couplé.** Une pénalité $\lambda\|\theta\|^2/2$ ajoute $\lambda\theta$ au gradient de données avant de mettre à jour les moments d’Adam. Quelle devient sa première composante ? Conserve-t-elle son signe ?

**2. Comparez avec AdamW.** Ses moments sont calculés à partir du seul gradient de données. La contraction des anciens paramètres, de facteur $1-\eta\lambda$, est appliquée séparément de la mise à jour adaptative. En utilisant le comportement du premier pas corrigé d’Adam, prévoyez les deux composantes finales dans les trois cas : Adam sans régularisation, Adam avec pénalité quadratique, AdamW.

**3. Exécutez le tableau de contrôle**, puis expliquez une différence observée en indiquant précisément à quel endroit du calcul intervient $\lambda$. La correction du démarrage et la décroissance des poids ne doivent pas être confondues.


In [ ]:
from IPython.display import display, Markdown
initial = torch.tensor([1., 2.], dtype=torch.float64)
data_gradient = torch.tensor([-.05, .5], dtype=torch.float64)
regularization = .1
rows = []
for method in ["Adam sans régularisation", "Adam + pénalité L2", "AdamW"]:
    theta = nn.Parameter(initial.clone())
    if method == "AdamW":
        opt = torch.optim.AdamW([theta], lr=.1, betas=(.9, .999), eps=1e-8, weight_decay=regularization)
        theta.grad = data_gradient.clone()
    else:
        opt = torch.optim.Adam([theta], lr=.1, betas=(.9, .999), eps=1e-8, weight_decay=0.)
        theta.grad = data_gradient.clone()
        if method == "Adam + pénalité L2":
            theta.grad += regularization * theta.detach()
    opt.step()
    rows.append((method, theta.detach().tolist()))
lines = ["| Méthode | θ₁ après un pas | θ₂ après un pas |", "|---|---:|---:|"]
lines.extend(f"| {name} | {value[0]:.6f} | {value[1]:.6f} |" for name, value in rows)
display(Markdown("\n".join(lines)))


#### Votre trace de travail

**Gradient couplé et prévision du premier pas** — À compléter.

**Différence entre les trois règles** — À compléter.


### Bilan du complément — Décrire un optimiseur au-delà de son nom

Pour terminer, expliquez le rôle du premier moment, du second moment non centré, de leurs corrections de démarrage et de la décroissance des poids. Écrivez ensuite les informations qu’un autre binôme devrait connaître pour reproduire votre comparaison : le seul nom « Adam » ne suffit pas.


#### Votre trace de travail

**Mémoires, corrections et informations à consigner** — À compléter.


## Sources et documentation pour poursuivre

- F. J. Anscombe, [*Graphs in Statistical Analysis*](https://doi.org/10.1080/00031305.1973.10478966), *The American Statistician* 27(1), 1973 : quatre jeux de données aux statistiques proches, conçus pour montrer la nécessité de représenter les observations.
- PyTorch, [Tensor views](https://docs.pytorch.org/docs/stable/tensor_view.html) et [broadcasting semantics](https://docs.pytorch.org/docs/stable/notes/broadcasting.html) : formes, indexation et diffusion des dimensions.
- PyTorch, [CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) : logits et format des cibles.
- PyTorch, [Autograd mechanics](https://docs.pytorch.org/docs/stable/notes/autograd.html) : graphe de calcul, accumulation des gradients et mécanismes d’évaluation.
- PyTorch, [gradcheck](https://docs.pytorch.org/docs/stable/generated/torch.autograd.gradcheck.html) : contrôle numérique des dérivées en double précision.
- D. P. Kingma et J. Ba, [*Adam: A Method for Stochastic Optimization*](https://arxiv.org/abs/1412.6980), ICLR 2015.
- I. Loshchilov et F. Hutter, [*Decoupled Weight Decay Regularization*](https://openreview.net/forum?id=Bkg6RiCqY7), ICLR 2019.
- M. Raissi, P. Perdikaris et G. E. Karniadakis, [*Physics-informed neural networks*](https://doi.org/10.1016/j.jcp.2018.10.045), *Journal of Computational Physics* 378, 2019.
- A. S. Krishnapriyan et al., [*Characterizing possible failure modes in physics-informed neural networks*](https://proceedings.neurips.cc/paper/2021/hash/df438e5206f31600e6ae4af72f2725f1-Abstract.html), NeurIPS 2021.
- S. Wang et al., [*An Expert’s Guide to Training Physics-informed Neural Networks*](https://arxiv.org/abs/2308.08468), 2023.

Les API PyTorch sont documentées en ligne et peuvent évoluer. Pour une expérience reproductible, consignez les versions effectivement utilisées, les graines, le matériel, le budget et les options des optimiseurs.


In [ ]:
print(f"Temps écoulé depuis la préparation du notebook : {time.perf_counter() - NOTEBOOK_START:.1f} s")
print("En usage interactif, cette durée inclut les lectures et les échanges ; ce n'est pas un temps de calcul pur.")
